# NucleiSky2D App
---

**NucleiSky** is a constellation-based registration framework designed to align microscopy images by matching the spatial arrangement of cell nuclei. It estimates a 2D **similarity transform** (rotation, uniform scale, and translation) between images, robustly handling challenging conditions such as partial overlap, missing nuclei, segmentation noise, or differing magnifications.

Instead of relying on pixel-intensity correlation, NucleiSky treats nuclei as a "constellation" of biological landmarks. It searches for **geometrically consistent correspondences** between these constellations, making it highly effective for cross-modality or cross-magnification alignment.

## Core Capabilities

* **Constellation Matching:** Aligns images using the relative geometry of nuclei centroids, independent of pixel intensity.
* **Scale Invariance:** Normalizes image scales prior to matching, allowing alignment of images acquired at different magnifications.
* **Robustness:** Uses RANSAC-style consensus to filter out outliers (e.g., debris, segmentation errors) and identify the coherent subset of nuclei that supports the correct transformation.
* **Modular Pipeline:** Components for segmentation, feature extraction, and matching can be used end-to-end or independently.

## The Matching Pipeline

NucleiSky operates in distinct stages to extract features, compute alignment, and apply results:

### Step 1: Preprocessing & Segmentation

To extract landmarks, the framework segments nuclei in both the reference and moving (crop) images. It supports **scale normalization** to ensure features are extracted at a consistent resolution.

**Available Segmentation Engines:**

* **Cellpose:** Strong general-purpose deep learning segmentation.
* **InstanSeg:** Fast, deep-learning-based segmentation optimized for efficiency; includes a tiled mode for large images.
* **Auto-Thresholding:** Classical computer vision (Otsu, Li, etc.) for high-contrast images where deep learning dependencies are unavailable.

*Note: The pipeline can also accept user-supplied binary masks.*

### Step 2 - Option A: Matching Strategies

NucleiSky includes four distinct geometric matchers and an "Adaptive" meta-strategy:

* **Graph-Based Matcher:** Constructs a graph (via Delaunay triangulation or k-NN) and matches local neighborhoods. This preserves **local topology**, making it robust when individual nuclei are ambiguous but the local cluster shape is distinctive.
* **Quad-Based Matcher:** Uses 4-point configurations (quadrilaterals) to create discriminative geometric primitives. Quads are highly specific, reducing false positives in dense or repetitive tissues.
* **Triangle-Based Matcher:** Matches 3-point constellations. While less discriminative than Quads, triangles are abundant and statistically robust, providing a strong fallback for sparse or noisy data.
* **Geometric Hashing:** An invariant voting system that indexes geometric coordinates into a hash table. This method accumulates evidence for a transform and is particularly effective for very dense constellations where graph matching might be too slow.

### Step 2 - Option B: Adaptive Mode (Auto-Pilot)

The framework features an **Adaptive Matching** strategy (`run_adaptive_nucleisky`) that automatically selects the optimal matcher order based on the density of nuclei:

* **Sparse Constellations:** Prioritizes **Quads** and **Triangles** for geometric rigidity.
* **Dense Constellations:** Prioritizes **Geometric Hashing** for efficiency.
* **Medium Density:** Uses a balanced approach starting with **Triangles** or **Graph** matching.

This mode iteratively tests strategies until a statistically confident match is found.

### Step 3: Applying Transforms (Reuse)

Once a transformation is computed (e.g., using the DAPI channel), it can be saved and **re-applied** to other images. This is essential for multi-channel workflows:

* **Cross-Channel Alignment:** Compute the alignment on nuclei, then apply the exact same transform to align other channels (e.g., GFP, RFP) that lack distinct landmarks.
* **Batch Processing:** Apply a validated transform to a time-lapse series or Z-stack.
* **Export Options:** The framework supports exporting aligned images in standard formats like **TIFF** or high-performance formats like **OME-Zarr** for large multi-dimensional datasets.

## Outputs

The pipeline outputs the optimal **similarity transform parameters** (Rotation matrix, Scale factor, Translation vector) and provides utility functions to visualize the alignment by overlaying the transformed crop onto the reference image.

# Step 0: Initialize Environment & Install Dependencies
---


**Run the following cell to initialize the environment.**

* **For Google Colab:** The notebook will automatically detect the environment, mount Google Drive, and install the necessary libraries.

* **For Local Jupyter Lab:** The notebook will skip installation steps to avoid messing up your local environment.



In [ ]:
# @title  Initialize Environment & Install Dependencies

from IPython.display import display
import sys

# Metadata
current_version = "0.0.8"
notebook_name = "NucleiSky2DApp"

# ---------------------------------------------------------
# 1. Environment Detection & Installation (Colab Only)
# ---------------------------------------------------------

if 'google.colab' in sys.modules:
    print("🚀 Detected Google Colab. Starting installation...")

    from google.colab import userdata

    # 1. Retrieve the token securely from Colab Secrets
    try:
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        print("Error: Secret 'GITHUB_TOKEN' not found. Please add it in the sidebar (🔑).")
        token = None

    # 2. Install from your private repo
    if token:
        user = 'CellMigrationLab'  # Replace with your GitHub username
        repo = 'NucleiSky-test'

        # Incompatibilities due to the newest scikit-image==0.26.0 version
        !pip install -q cucim-cu12==26.4.0
        # We use the PEP 508 syntax: "package[extras] @ url"
        !pip install -q "nucleisky[all] @ git+https://{token}@github.com/{user}/{repo}.git"

    from google.colab import drive
    drive.mount('/content/gdrive')

    print("✅ Colab setup done")

else:
    # Fallback for local environments
    print("✅ Local environment detected. Assuming dependencies are already installed.")


# ---------------------------------------------------------
# 2. GPU Check
# ---------------------------------------------------------
try:
    import torch
    if torch.cuda.is_available():
            print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
    else:
        print("⚠️  WARNING: No GPU detected. Segmentation with Cellpose or InstanSeg will be VERY SLOW on CPU.")
        if 'google.colab' in sys.modules:
            print("Go to Runtime > Change runtime type > Hardware accelerator > T4 GPU")
except ImportError:
    print("⚠️  WARNING: PyTorch not installed. Cellpose and InstanSeg segmentation will not work. If you want to use these features, please check the installation instructions.")

# ---------------------------------------------------------
# 3. Main Imports
# ---------------------------------------------------------

from IPython.display import display, clear_output
from tifffile import imwrite, imread
import ipywidgets as widgets
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import traceback
import html
import copy
import math

from nucleisky2d.demo_utils import generate_random_crop

# IO
from nucleisky2d.io import (
    load_image,
    load_transforms_any,
    get_pixel_size_um_from_tiff,
    save_json,
    make_result_dir,
    save_nucleisky_transform,
    append_transform_jsonl
)

# Preprocess
from nucleisky2d.preprocess import (
    require_2d_image,
    require_2d_label_mask,
    require_positive_float,
    scale_normalize_pair_for_segmentation,
    ij_percentile_normalize
)


# 1. Pure Export
from nucleisky2d.export import (
    export_aligned_imagej_stacks,
    run_export_with_record,
    inspect_image_header,
)

# 2. Visualization
from nucleisky2d.visualization import (
    imshow_safe,
    show_alignment_original_and_rescaled,
    downsample_points_for_display
)

# -----------------------------------------------

from nucleisky2d.segmentation import segment_nuclei_dispatch
from nucleisky2d.features import (
    extract_nuclear_features,
    add_centroids_orig_px_columns,
    centroids_from_df,
    extract_centroids_um,
    stack_feature_vectors
)
from nucleisky2d.config import save_matcher_config_json, DEFAULT_MATCHER_CONFIG
from nucleisky2d.pipeline import (
    NucleiSky,
    run_adaptive_matching_and_export,
    pick_best_transform
)


# ----------------------------
# UI Styling & CSS
# ----------------------------
STYLE_CSS = """
<style>
:root {
    --ns-bg:              #FFFFFF;
    --ns-bg-page:         #F1F5F9;
    --ns-bg-soft:         #F8FAFC;
    --ns-bg-softer:       #F9FAFB;
    --ns-border:          #CBD5E1;
    --ns-border-soft:     #E2E8F0;
    --ns-text:            #1E293B;
    --ns-text-muted:      #334155;
    --ns-text-soft:       #475569;
    --ns-label:           #475569;

    --ns-ok-bg:           #ECFDF5;
    --ns-ok-border:       #6EE7B7;
    --ns-ok-text:         #065F46;

    --ns-err-bg:          #FEF2F2;
    --ns-err-border:      #FCA5A5;
    --ns-err-text:        #991B1B;

    --ns-warn-bg:         #FFFBEB;
    --ns-warn-border:     #FCD34D;
    --ns-warn-text:       #92400E;

    --ns-accent:          #2563EB;
    --ns-accent-hover:    #1D4ED8;
    --ns-accent-soft:     #DBEAFE;

    --ns-info-bg:         #0284C7;
    --ns-info-hover:      #0369A1;
    --ns-info-text:       #FFFFFF;

    --ns-spinner-track:   #BFDBFE;

    --ns-btn-bg:          #F1F5F9;
    --ns-btn-hover-bg:    #E2E8F0;
    --ns-btn-text:        #334155;
    --ns-btn-border:      #CBD5E1;
    --ns-btn-active-bg:   #2563EB;
    --ns-btn-active-text: #FFFFFF;

    --ns-disabled-bg:     #EFF6FF;
    --ns-disabled-text:   #1E40AF;
    --ns-disabled-border: #BFDBFE;

    --ns-disabled-strong-bg:     #DBEAFE;
    --ns-disabled-strong-text:   #1E3A8A;
    --ns-disabled-strong-border: #93C5FD;
}

/* ---------------------------------------------------------
   Base layout
--------------------------------------------------------- */

.ns-root {
    background-color: var(--ns-bg-page) !important;
    padding: 20px;
    border-radius: 10px;
    color: var(--ns-text) !important;
    color-scheme: light !important;
}

.ns-card {
    border: 1px solid var(--ns-border);
    border-radius: 8px;
    padding: 16px;
    margin-bottom: 16px;
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    box-shadow: 0 1px 3px rgba(0,0,0,0.06);
    color-scheme: light !important;
}

.ns-card-warn {
    border: 1px solid var(--ns-err-border);
    background-color: var(--ns-err-bg) !important;
}

.ns-header {
    font-weight: 700;
    font-size: 13px;
    color: var(--ns-text-soft) !important;
    margin-bottom: 12px;
    text-transform: uppercase;
    letter-spacing: 0.06em;
    border-bottom: 1px solid var(--ns-border-soft);
    padding-bottom: 8px;
}

.ns-label {
    font-size: 11px;
    font-weight: 700;
    color: var(--ns-label) !important;
    margin-top: 8px;
    margin-bottom: 4px;
    text-transform: uppercase;
    letter-spacing: 0.04em;
}

.ns-page-title {
    font-weight: 800;
    font-size: 18px;
    color: var(--ns-text) !important;
    margin: 0 0 4px 0;
}

.ns-page-subtitle {
    font-size: 13px;
    color: var(--ns-text-muted) !important;
    margin-bottom: 16px;
}

.widget-label {
    color: var(--ns-text-muted) !important;
    font-weight: 500 !important;
}

.widget-label-basic {
    color: var(--ns-text-muted) !important;
}

.ns-subhead {
    font-size: 11px;
    font-weight: 700;
    color: var(--ns-label) !important;
    text-transform: uppercase;
    letter-spacing: 0.06em;
    margin: 8px 0 6px 0;
}

.ns-body-text {
    font-size: 12px;
    color: var(--ns-text-soft) !important;
    line-height: 1.5;
}

.ns-body-text b,
.ns-body-text strong {
    color: var(--ns-text) !important;
}

.ns-body-text ul {
    margin: 6px 0 0 18px;
    padding: 0;
}

.ns-body-text li {
    margin: 2px 0;
}

.ns-desc {
    font-size: 12px;
    color: var(--ns-text-soft) !important;
    margin-bottom: 8px;
    font-style: italic;
}

/* ---------------------------------------------------------
   Status messages
--------------------------------------------------------- */

.ns-status {
    font-size: 13px;
    font-weight: 600;
    margin: 8px 0;
}

.ns-status-running {
    display: flex;
    align-items: center;
    gap: 10px;
    color: var(--ns-accent) !important;
}

.ns-status-ok {
    color: var(--ns-ok-text) !important;
}

.ns-status-warn {
    color: var(--ns-warn-text) !important;
}

.ns-status-error {
    color: var(--ns-err-text) !important;
}

.ns-message {
    width: 99% !important;
    box-sizing: border-box !important;
    border-radius: 8px !important;
    padding: 14px 16px !important;
    margin: 12px 0 !important;
    line-height: 1.45 !important;
    color-scheme: light !important;
}

.ns-message-title {
    font-weight: 800 !important;
    font-size: 14px !important;
    margin-bottom: 6px !important;
}

.ns-message-body {
    font-size: 13px !important;
    line-height: 1.45 !important;
}

.ns-message-error {
    border: 1px solid var(--ns-err-border) !important;
    background-color: var(--ns-err-bg) !important;
    color: var(--ns-err-text) !important;
}

.ns-message-error .ns-message-title,
.ns-message-error .ns-message-body {
    color: var(--ns-err-text) !important;
}

.ns-message-ok {
    border: 1px solid var(--ns-ok-border) !important;
    background-color: var(--ns-ok-bg) !important;
    color: var(--ns-ok-text) !important;
}

.ns-message-ok .ns-message-title,
.ns-message-ok .ns-message-body {
    color: var(--ns-ok-text) !important;
}

.ns-message-warn {
    border: 1px solid var(--ns-warn-border) !important;
    background-color: var(--ns-warn-bg) !important;
    color: var(--ns-warn-text) !important;
}

.ns-message-warn .ns-message-title,
.ns-message-warn .ns-message-body {
    color: var(--ns-warn-text) !important;
}

.ns-spinner {
    width: 14px;
    height: 14px;
    border: 2px solid var(--ns-spinner-track);
    border-top: 2px solid var(--ns-accent);
    border-radius: 50%;
    animation: ns-spin 0.8s linear infinite;
}

.ns-step-active {
    background: var(--ns-spinner-track) !important;
    color: var(--ns-accent) !important;
}

.ns-step-idle {
    background: var(--ns-bg-soft) !important;
    color: var(--ns-text-soft) !important;
}

/* ---------------------------------------------------------
   Toggle buttons
--------------------------------------------------------- */

.widget-toggle-buttons .widget-toggle-button {
    background: var(--ns-btn-bg) !important;
    color: var(--ns-btn-text) !important;
    border: 1px solid var(--ns-btn-border) !important;
}

.widget-toggle-buttons .widget-toggle-button.mod-active {
    background: var(--ns-btn-active-bg) !important;
    color: var(--ns-btn-active-text) !important;
    border-color: var(--ns-btn-active-bg) !important;
}

/* ---------------------------------------------------------
   General widget light-theme forcing inside cards
--------------------------------------------------------- */

.ns-card .widget-box,
.ns-card .widget-hbox,
.ns-card .widget-vbox,
.ns-card .widget-gridbox,
.ns-card .widget-label,
.ns-card .widget-label-basic,
.ns-card .widget-html,
.ns-card .widget-output,
.ns-card .jupyter-widgets,
.ns-card .p-Widget,
.ns-card .lm-Widget {
    background-color: transparent !important;
    color: var(--ns-text) !important;
    color-scheme: light !important;
}

.ns-card select,
.ns-card input,
.ns-card textarea,
.ns-card button {
    color-scheme: light !important;
}

.ns-card select,
.ns-card input,
.ns-card textarea {
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    border-color: var(--ns-border) !important;
}

.ns-card select:disabled,
.ns-card input:disabled,
.ns-card textarea:disabled {
    background-color: var(--ns-bg-soft) !important;
    color: var(--ns-text-soft) !important;
    border-color: var(--ns-border-soft) !important;
    opacity: 1 !important;
}

/* ---------------------------------------------------------
   Button hard fix
   The flex rules here keep the label vertically centered.
--------------------------------------------------------- */

.ns-card .widget-button,
.ns-root .widget-button,
.ns-tabs .widget-button,
.ns-tab-panel .widget-button,
.ns-accordion .widget-button,
.ns-accordion-body .widget-button,
button.widget-button {
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    gap: 6px !important;

    box-sizing: border-box !important;
    line-height: 1 !important;
    text-align: center !important;
    vertical-align: middle !important;

    background-color: var(--ns-btn-bg) !important;
    color: var(--ns-btn-text) !important;
    border: 1px solid var(--ns-btn-border) !important;
    border-radius: 6px !important;
    font-weight: 700 !important;
    box-shadow: 0 1px 2px rgba(15,23,42,0.10) !important;

    min-height: 32px !important;
    padding: 0 12px !important;
    cursor: pointer !important;
    opacity: 1 !important;
    color-scheme: light !important;
}

.ns-card .widget-button i,
.ns-root .widget-button i,
.ns-tabs .widget-button i,
.ns-tab-panel .widget-button i,
.ns-accordion .widget-button i,
.ns-accordion-body .widget-button i,
button.widget-button i,
.ns-card .widget-button .fa,
.ns-root .widget-button .fa,
.ns-tabs .widget-button .fa,
.ns-tab-panel .widget-button .fa,
.ns-accordion .widget-button .fa,
.ns-accordion-body .widget-button .fa,
button.widget-button .fa {
    line-height: 1 !important;
    margin: 0 !important;
    padding: 0 !important;
    display: inline-flex !important;
    align-items: center !important;
}

.ns-card .widget-button:hover,
.ns-root .widget-button:hover,
.ns-tabs .widget-button:hover,
.ns-tab-panel .widget-button:hover,
.ns-accordion .widget-button:hover,
.ns-accordion-body .widget-button:hover,
button.widget-button:hover {
    background-color: var(--ns-btn-hover-bg) !important;
    color: var(--ns-text) !important;
    border-color: var(--ns-border) !important;
}

.ns-card .widget-button.mod-info,
.ns-root .widget-button.mod-info,
.ns-tabs .widget-button.mod-info,
.ns-tab-panel .widget-button.mod-info,
.ns-accordion .widget-button.mod-info,
.ns-accordion-body .widget-button.mod-info,
button.widget-button.mod-info,
.ns-action-button {
    background-color: var(--ns-info-bg) !important;
    color: var(--ns-info-text) !important;
    border-color: var(--ns-info-bg) !important;
    box-shadow: 0 2px 4px rgba(2,132,199,0.22) !important;
}

.ns-card .widget-button.mod-info:hover,
.ns-root .widget-button.mod-info:hover,
.ns-tabs .widget-button.mod-info:hover,
.ns-tab-panel .widget-button.mod-info:hover,
.ns-accordion .widget-button.mod-info:hover,
.ns-accordion-body .widget-button.mod-info:hover,
button.widget-button.mod-info:hover,
.ns-action-button:hover {
    background-color: var(--ns-info-hover) !important;
    color: var(--ns-info-text) !important;
    border-color: var(--ns-info-hover) !important;
}

.ns-card .widget-button.mod-primary,
.ns-root .widget-button.mod-primary,
.ns-tabs .widget-button.mod-primary,
.ns-tab-panel .widget-button.mod-primary,
.ns-accordion .widget-button.mod-primary,
.ns-accordion-body .widget-button.mod-primary,
button.widget-button.mod-primary,
.ns-primary-button {
    background-color: var(--ns-accent) !important;
    color: #FFFFFF !important;
    border-color: var(--ns-accent) !important;
    box-shadow: 0 2px 4px rgba(37,99,235,0.22) !important;
}

.ns-card .widget-button.mod-primary:hover,
.ns-root .widget-button.mod-primary:hover,
.ns-tabs .widget-button.mod-primary:hover,
.ns-tab-panel .widget-button.mod-primary:hover,
.ns-accordion .widget-button.mod-primary:hover,
.ns-accordion-body .widget-button.mod-primary:hover,
button.widget-button.mod-primary:hover,
.ns-primary-button:hover {
    background-color: var(--ns-accent-hover) !important;
    color: #FFFFFF !important;
    border-color: var(--ns-accent-hover) !important;
}

.ns-card .widget-button.mod-success,
.ns-root .widget-button.mod-success,
button.widget-button.mod-success {
    background-color: var(--ns-ok-text) !important;
    color: #FFFFFF !important;
    border-color: var(--ns-ok-text) !important;
}

.ns-card .widget-button.mod-warning,
.ns-root .widget-button.mod-warning,
button.widget-button.mod-warning {
    background-color: #D97706 !important;
    color: #FFFFFF !important;
    border-color: #D97706 !important;
}

.ns-card .widget-button.mod-danger,
.ns-root .widget-button.mod-danger,
button.widget-button.mod-danger {
    background-color: var(--ns-err-text) !important;
    color: #FFFFFF !important;
    border-color: var(--ns-err-text) !important;
}

/* Disabled buttons: visible but clearly disabled */
.ns-card .widget-button:disabled,
.ns-root .widget-button:disabled,
.ns-tabs .widget-button:disabled,
.ns-tab-panel .widget-button:disabled,
.ns-accordion .widget-button:disabled,
.ns-accordion-body .widget-button:disabled,
button.widget-button:disabled {
    background-color: var(--ns-disabled-bg) !important;
    color: var(--ns-disabled-text) !important;
    border-color: var(--ns-disabled-border) !important;
    box-shadow: none !important;
    cursor: not-allowed !important;
    opacity: 1 !important;
}

/* Disabled primary buttons should remain recognisable */
.ns-card .widget-button.mod-primary:disabled,
.ns-root .widget-button.mod-primary:disabled,
.ns-tabs .widget-button.mod-primary:disabled,
.ns-tab-panel .widget-button.mod-primary:disabled,
.ns-accordion .widget-button.mod-primary:disabled,
.ns-accordion-body .widget-button.mod-primary:disabled,
button.widget-button.mod-primary:disabled,
.ns-primary-button:disabled {
    background-color: var(--ns-disabled-strong-bg) !important;
    color: var(--ns-disabled-strong-text) !important;
    border-color: var(--ns-disabled-strong-border) !important;
    font-weight: 800 !important;
    opacity: 1 !important;
}

/* ---------------------------------------------------------
   Accordion hard fix for Colab dark theme
--------------------------------------------------------- */

.ns-accordion,
.ns-accordion-body {
    color-scheme: light !important;
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
}

.widget-accordion.ns-accordion,
.ns-accordion.widget-accordion,
.ns-card .widget-accordion,
.widget-accordion.ns-card {
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    border: 1px solid var(--ns-border-soft) !important;
    border-radius: 8px !important;
    overflow: hidden !important;
    color-scheme: light !important;
}

.ns-accordion .p-AccordionPanel,
.ns-accordion .lm-AccordionPanel,
.ns-card .p-AccordionPanel,
.ns-card .lm-AccordionPanel {
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    border: none !important;
    color-scheme: light !important;
}

.ns-accordion .p-AccordionPanel-title,
.ns-accordion .lm-AccordionPanel-title,
.ns-card .p-AccordionPanel-title,
.ns-card .lm-AccordionPanel-title {
    background-color: var(--ns-bg-soft) !important;
    color: var(--ns-text) !important;
    border-bottom: 1px solid var(--ns-border-soft) !important;
    font-weight: 700 !important;
    min-height: 32px !important;
    padding: 6px 10px !important;
    color-scheme: light !important;
}

.ns-accordion .p-AccordionPanel-title:hover,
.ns-accordion .lm-AccordionPanel-title:hover,
.ns-accordion .p-AccordionPanel-title.p-mod-current,
.ns-accordion .lm-AccordionPanel-title.lm-mod-current,
.ns-card .p-AccordionPanel-title:hover,
.ns-card .lm-AccordionPanel-title:hover,
.ns-card .p-AccordionPanel-title.p-mod-current,
.ns-card .lm-AccordionPanel-title.lm-mod-current {
    background-color: var(--ns-bg-soft) !important;
    color: var(--ns-text) !important;
}

.ns-accordion .p-AccordionPanel-titleLabel,
.ns-accordion .lm-AccordionPanel-titleLabel,
.ns-card .p-AccordionPanel-titleLabel,
.ns-card .lm-AccordionPanel-titleLabel {
    color: var(--ns-text) !important;
}

.ns-accordion .p-AccordionPanel-titleIcon,
.ns-accordion .lm-AccordionPanel-titleIcon,
.ns-card .p-AccordionPanel-titleIcon,
.ns-card .lm-AccordionPanel-titleIcon {
    color: var(--ns-text-soft) !important;
}

.ns-accordion .p-AccordionPanel-widget,
.ns-accordion .lm-AccordionPanel-widget,
.ns-accordion .p-AccordionPanel-content,
.ns-accordion .lm-AccordionPanel-content,
.ns-card .p-AccordionPanel-widget,
.ns-card .lm-AccordionPanel-widget,
.ns-card .p-AccordionPanel-content,
.ns-card .lm-AccordionPanel-content {
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    padding: 12px !important;
    color-scheme: light !important;
}

.ns-accordion-body,
.widget-box.ns-accordion-body,
.widget-vbox.ns-accordion-body,
.widget-hbox.ns-accordion-body,
.ns-accordion-body.widget-box,
.ns-accordion-body.widget-vbox,
.ns-accordion-body.widget-hbox,
.ns-accordion-body.p-Widget,
.ns-accordion-body.lm-Widget {
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    color-scheme: light !important;
}

.ns-accordion-body .widget-box,
.ns-accordion-body .widget-vbox,
.ns-accordion-body .widget-hbox,
.ns-accordion-body .widget-gridbox,
.ns-accordion-body .widget-label,
.ns-accordion-body .widget-label-basic,
.ns-accordion-body .widget-html,
.ns-accordion-body .widget-output,
.ns-accordion-body .jupyter-widgets,
.ns-accordion-body .p-Widget,
.ns-accordion-body .lm-Widget {
    background-color: transparent !important;
    color: var(--ns-text) !important;
    color-scheme: light !important;
}

.ns-accordion-body select,
.ns-accordion-body input,
.ns-accordion-body textarea,
.ns-accordion-body button,
.ns-accordion select,
.ns-accordion input,
.ns-accordion textarea,
.ns-accordion button {
    color-scheme: light !important;
}

.ns-accordion-body select,
.ns-accordion-body input,
.ns-accordion-body textarea,
.ns-accordion select,
.ns-accordion input,
.ns-accordion textarea {
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    border-color: var(--ns-border) !important;
}

.ns-accordion-body select:disabled,
.ns-accordion-body input:disabled,
.ns-accordion-body textarea:disabled,
.ns-accordion select:disabled,
.ns-accordion input:disabled,
.ns-accordion textarea:disabled {
    background-color: var(--ns-bg-soft) !important;
    color: var(--ns-text-soft) !important;
    border-color: var(--ns-border-soft) !important;
    opacity: 1 !important;
}

/* ---------------------------------------------------------
   Tab hard fix for Colab dark theme
--------------------------------------------------------- */

.ns-tabs,
.ns-tab-panel {
    color-scheme: light !important;
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
}

.widget-tab.ns-tabs,
.ns-tabs.widget-tab,
.ns-card .widget-tab {
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    border: 1px solid var(--ns-border-soft) !important;
    border-radius: 8px !important;
    overflow: hidden !important;
    color-scheme: light !important;
}

.ns-tabs .p-TabPanel,
.ns-tabs .lm-TabPanel,
.ns-card .p-TabPanel,
.ns-card .lm-TabPanel {
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    color-scheme: light !important;
}

.ns-tabs .p-TabBar,
.ns-tabs .lm-TabBar,
.ns-tabs .p-TabBar-content,
.ns-tabs .lm-TabBar-content,
.ns-card .p-TabBar,
.ns-card .lm-TabBar,
.ns-card .p-TabBar-content,
.ns-card .lm-TabBar-content {
    background-color: var(--ns-bg-soft) !important;
    color: var(--ns-text) !important;
    border-bottom: 1px solid var(--ns-border-soft) !important;
    color-scheme: light !important;
}

.ns-tabs .p-TabBar-tab,
.ns-tabs .lm-TabBar-tab,
.ns-card .p-TabBar-tab,
.ns-card .lm-TabBar-tab {
    background-color: var(--ns-bg-soft) !important;
    color: var(--ns-text-soft) !important;
    border: none !important;
    border-right: 1px solid var(--ns-border-soft) !important;
    padding: 8px 12px !important;
    min-height: 34px !important;
    font-weight: 700 !important;
    color-scheme: light !important;
}

.ns-tabs .p-TabBar-tab.p-mod-current,
.ns-tabs .lm-TabBar-tab.lm-mod-current,
.ns-card .p-TabBar-tab.p-mod-current,
.ns-card .lm-TabBar-tab.lm-mod-current {
    background-color: var(--ns-bg) !important;
    color: var(--ns-accent) !important;
    border-bottom: 2px solid var(--ns-accent) !important;
}

.ns-tabs .p-TabBar-tab:hover,
.ns-tabs .lm-TabBar-tab:hover,
.ns-card .p-TabBar-tab:hover,
.ns-card .lm-TabBar-tab:hover {
    background-color: var(--ns-accent-soft) !important;
    color: var(--ns-accent) !important;
}

.ns-tabs .p-TabBar-tabLabel,
.ns-tabs .lm-TabBar-tabLabel,
.ns-tabs .p-TabBar-tabIcon,
.ns-tabs .lm-TabBar-tabIcon,
.ns-tabs .p-TabBar-tabCloseIcon,
.ns-tabs .lm-TabBar-tabCloseIcon,
.ns-card .p-TabBar-tabLabel,
.ns-card .lm-TabBar-tabLabel,
.ns-card .p-TabBar-tabIcon,
.ns-card .lm-TabBar-tabIcon,
.ns-card .p-TabBar-tabCloseIcon,
.ns-card .lm-TabBar-tabCloseIcon {
    color: inherit !important;
}

.ns-tabs .p-TabPanel-stackedPanel,
.ns-tabs .lm-TabPanel-stackedPanel,
.ns-tabs .p-StackedPanel,
.ns-tabs .lm-StackedPanel,
.ns-card .p-TabPanel-stackedPanel,
.ns-card .lm-TabPanel-stackedPanel,
.ns-card .p-StackedPanel,
.ns-card .lm-StackedPanel {
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    padding: 0 !important;
    color-scheme: light !important;
}

.ns-tab-panel,
.widget-box.ns-tab-panel,
.widget-vbox.ns-tab-panel,
.widget-hbox.ns-tab-panel,
.ns-tab-panel.widget-box,
.ns-tab-panel.widget-vbox,
.ns-tab-panel.widget-hbox,
.ns-tab-panel.p-Widget,
.ns-tab-panel.lm-Widget {
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    padding: 12px !important;
    color-scheme: light !important;
}

.ns-tab-panel .widget-box,
.ns-tab-panel .widget-vbox,
.ns-tab-panel .widget-hbox,
.ns-tab-panel .widget-gridbox,
.ns-tab-panel .widget-label,
.ns-tab-panel .widget-label-basic,
.ns-tab-panel .widget-html,
.ns-tab-panel .widget-output,
.ns-tab-panel .jupyter-widgets,
.ns-tab-panel .p-Widget,
.ns-tab-panel .lm-Widget {
    background-color: transparent !important;
    color: var(--ns-text) !important;
    color-scheme: light !important;
}

.ns-tab-row {
    background-color: transparent !important;
    color: var(--ns-text) !important;
    border-bottom: 1px solid var(--ns-border-soft);
    padding: 6px 0;
    color-scheme: light !important;
}

.ns-tab-row:last-child {
    border-bottom: none;
}

.ns-tabs select,
.ns-tabs input,
.ns-tabs textarea,
.ns-tabs button,
.ns-tab-panel select,
.ns-tab-panel input,
.ns-tab-panel textarea,
.ns-tab-panel button {
    color-scheme: light !important;
}

.ns-tabs select,
.ns-tabs input,
.ns-tabs textarea,
.ns-tab-panel select,
.ns-tab-panel input,
.ns-tab-panel textarea {
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    border-color: var(--ns-border) !important;
}

.ns-tabs select:disabled,
.ns-tabs input:disabled,
.ns-tabs textarea:disabled,
.ns-tab-panel select:disabled,
.ns-tab-panel input:disabled,
.ns-tab-panel textarea:disabled {
    background-color: var(--ns-bg-soft) !important;
    color: var(--ns-text-soft) !important;
    border-color: var(--ns-border-soft) !important;
    opacity: 1 !important;
}

/* ---------------------------------------------------------
   Animation
--------------------------------------------------------- */

@keyframes ns-spin {
    0% {
        transform: rotate(0deg);
    }
    99% {
        transform: rotate(360deg);
    }
}
</style>
"""

# ----------------------------------------
# Temporal functions for logging

import logging

class OutputWidgetHandler(logging.Handler):
    def __init__(self, output_widget):
        super().__init__()
        self.output_widget = output_widget

    def emit(self, record):
        msg = self.format(record)
        with self.output_widget:
            print(msg)

def configure_notebook_logging(output_widget):
    logger = logging.getLogger("nucleisky2d")
    logger.setLevel(logging.INFO)

    # Avoid duplicate handlers if the cell is run multiple times
    logger.handlers = [
        h for h in logger.handlers
        if not isinstance(h, OutputWidgetHandler)
    ]

    handler = OutputWidgetHandler(output_widget)
    handler.setLevel(logging.INFO)
    handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))

    logger.addHandler(handler)
    logger.propagate = False
    return logger

def _spinner_html(text):
    return f"""
    <div class="ns-status ns-status-running">
        <div class="ns-spinner"></div>
        <div>{text}</div>
    </div>
    """

# ============================================
# Small UI / HTML helpers
# ============================================

def _status_message(lines, title="Status", kind="info"):
    """Return a full-width styled status message."""
    if isinstance(lines, str):
        lines = [lines]

    safe_lines = "<br>".join(html.escape(str(x)) for x in lines)

    class_map = {
        "ok": "ns-message-ok",
        "error": "ns-message-error",
        "warn": "ns-message-warn",
        "info": "ns-message-warn",
    }

    cls = class_map.get(kind, "ns-message-warn")

    return f"""
    <div class="ns-message {cls}">
      <div class="ns-message-title">{html.escape(str(title))}</div>
      <div class="ns-message-body">{safe_lines}</div>
    </div>
    """


def _status_ok(lines, title="Ready"):
    return _status_message(lines, title=title, kind="ok")


def _status_err(lines, title="Action required"):
    return _status_message(lines, title=title, kind="error")


def _status_warn(lines, title="Please check"):
    return _status_message(lines, title=title, kind="warn")


def _status_info(lines, title="Working"):
    return f"""
    <div class="ns-status ns-status-running">
      <div class="ns-spinner"></div>
      <div>{html.escape(str(title))}: {html.escape(str(lines))}</div>
    </div>
    """

def _status_html(text, kind="ok"):
    return f"<div class='ns-status ns-status-{kind}'>{text}</div>"

print("✅ Environment Ready")

# Step 1: Load your images and extract features
---

### **Select Images & Configuration**

In this step, you will define the input data for the matching process: the **Reference Image** (the "sky map") and the **Partial View** (the "telescope snapshot").

**1. Source Images**

* `Full Path`: Enter the file path to your large reference image.
* `Load External Crop`: Use a real experimental ROI image you want to locate.
* `Generate Random Crop`: Synthesize a crop from the reference image (useful for testing the pipeline without a separate ROI file).


**2. Random Generator Settings**
*(Only active if "Generate Random Crop" is selected)*

* `Output Size`: Dimensions of the synthetic crop in pixels.
* `Rotation`: Maximum rotation angle ( degrees) to apply.
* `Scaling Range`: Simulates magnification differences (e.g., `0.7`–`1.3` allows the crop to be 30% smaller or larger than the reference).
* `Fixed Seed`: Ensures the exact same random crop is generated every time for reproducibility.

**⚠️ The "Golden Rule": Pixel Size ()**
NucleiSky2D relies on physical units to match constellations across scales.

* The tool automatically reads metadata from TIFF headers.
* **Action Required:** If pixel size metadata is missing, a **red warning box** will appear. You **must** manually enter the correct  values for matching to succeed.

**Note on Image Dimensions:**
The pipeline performs registration on **2D nuclei centroids**.

* **RGB Images:** Accepted and converted to grayscale internally.
* **Z-Stacks / Time Series:** Not supported directly in this step. Please select a single representative plane (or maximum projection) before loading, or ensure your input path points to a 2D file.

In [ ]:
#@title Run to load image choosing GUI

# ----------------------------
# UI Styling
# ----------------------------

display(widgets.HTML(STYLE_CSS))

# ----------------------------
# Utilities
# ----------------------------

def _desc_width(*ws, w="90px"):
    for x in ws:
        try: x.style.description_width = w
        except: pass

# ------------------------------------------------------------
# State & Globals
# ------------------------------------------------------------
state = dict(
    confirmed_full=False,
    confirmed_crop=False,
    last_full_path="",
    last_crop_path="",
    need_full=False,
    need_crop=False,
)

# Prefill
_last_full = float(globals().get("_NUCLEISKY_LAST_MANUAL_FULL", 1.0))
_last_crop = float(globals().get("_NUCLEISKY_LAST_MANUAL_CROP", 1.0))

# ----------------------------
# WIDGETS CONSTRUCTION
# ----------------------------
style_html = widgets.HTML(STYLE_CSS)

# --- Header ---
title_html = widgets.HTML(
    "<div class='ns-page-title'>Choose Images and Output Folder</div>"
    "<div class='ns-page-subtitle'>Select your reference image, define the crop region (from an image or randomly generated) and choose the output folder.</div>"
)

# --- Card 1: Random Settings (Refactored Layout) (will appear inside the main card) ---
# Inputs
patch_h = widgets.IntText(value=200, description='Height (px)', layout=widgets.Layout(width='98%'))
patch_w = widgets.IntText(value=200, description='Width (px)', layout=widgets.Layout(width='98%'))
#  - Helps user visualize H/W logic
max_angle = widgets.FloatText(value=180, description='Max Angle (°)', layout=widgets.Layout(width='98%'))
#  - Helps user visualize rotation range
zoom_min = widgets.FloatText(value=0.7, description='Zoom Min', layout=widgets.Layout(width='98%'))
zoom_max = widgets.FloatText(value=1.3, description='Zoom Max', layout=widgets.Layout(width='98%'))

use_seed = widgets.Checkbox(value=False, description='Fixed Seed', indent=False)
seed = widgets.IntText(value=42, description='Seed Value', layout=widgets.Layout(width='220px'), disabled=True)

# New Layout Structure
rand_grid = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Random Generator Settings</div>"),

    # Section 1: Dimensions
    widgets.HTML("<div class='ns-label'>Output Size</div>"),
    widgets.HBox([patch_h, patch_w], layout=widgets.Layout(gap="10px")),

    # Section 2: Rotation (Moved here)
    widgets.HTML("<div class='ns-label'>Rotation</div>"),
    max_angle,

    # Section 3: Zoom
    widgets.HTML("<div class='ns-label'>Scaling Range</div>"),
    widgets.HBox([zoom_min, zoom_max], layout=widgets.Layout(gap="10px")),

    # Section 4: Reproducibility
    widgets.HTML("<div style='height:12px'></div>"),
    widgets.HBox([use_seed, seed], layout=widgets.Layout(gap="10px"))
])
rand_card = widgets.VBox([rand_grid])
rand_card.add_class("ns-card")

# --- Card 2: Main Inputs ---
img_path = widgets.Text(
    value='/content/gdrive/MyDrive/Work/manuscript/Ongoing Projects/NucleiSky/Datasets/microfluidic channel fixed/image1/Image1.tif',
    description='Full Path', placeholder='/path/to/large_image.tif', layout=widgets.Layout(width='98%')
)

mode = widgets.ToggleButtons(
    options=[("Load External Crop", "external"), ("Generate Random Crop", "random")],
    value="external",
    layout=widgets.Layout(width='auto'),
    style={'button_width':'180px'}
)

crop_path = widgets.Text(value='', description='Crop Path', placeholder='/path/to/matching_crop.tif', layout=widgets.Layout(width='98%'))

input_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Source Images</div>"),
    img_path,
    widgets.HTML("<div style='height:8px'></div>"),
    widgets.HTML("<div class='ns-label'>Crop Strategy</div>"),
    mode,
    widgets.HTML("<div style='height:8px'></div>"),
    crop_path,
    rand_card
])
input_card.add_class("ns-card")

# --- Card 3: Manual Metadata (Warning Card) ---
manual_msg = widgets.HTML("", layout=widgets.Layout(width="99%"))
manual_full = widgets.FloatText(value=_last_full, description='Full µm/px')
manual_crop = widgets.FloatText(value=_last_crop, description='Crop µm/px')

manual_card = widgets.VBox([
    widgets.HTML("<div class='ns-header' style='color:var(--ns-err-text); border-color:var(--ns-err-border);'>Missing Metadata</div>"),
    manual_msg,
    widgets.HBox([manual_full], layout=widgets.Layout(width="99%", margin="5px 0")),
    widgets.HBox([manual_crop], layout=widgets.Layout(width="99%", margin="5px 0"))
])
manual_card.add_class("ns-card")
manual_card.add_class("ns-card-warn")
manual_card.layout.display = "none"

# --- Card 4: Output Folder ---

output_dir = widgets.Text(
    value='',
    description='Folder',
    placeholder='/path/to/output_folder',
    layout=widgets.Layout(width='98%')
)

output_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Output Folder</div>"),
    widgets.HTML("<span style='font-size:13px; color:var(--ns-text-muted);'>Please select the folder to save the results or leave it empty and a default folder will be chosen.</span>"),
    output_dir,
])
output_card.add_class("ns-card")

# --- Outputs ---
status_html = widgets.HTML("", layout=widgets.Layout(width="99%"))
run_button = widgets.Button(description=' Load & Process', button_style='primary', icon='play', layout=widgets.Layout(width='99%', height='40px'))
fig_out = widgets.Output(layout=widgets.Layout(max_height="500px"))

# Styling widths
_desc_width(img_path, crop_path, w="80px")
_desc_width(patch_h, patch_w, zoom_min, zoom_max, max_angle, manual_full, manual_crop, w="90px")

# ----------------------------
# Logic & Sync
# ----------------------------
def _sync_ui():
    use_rand = (mode.value == "random")

    if use_rand:
        crop_path.layout.display = "none"
        rand_card.layout.display = "flex"
    else:
        crop_path.layout.display = "flex"
        rand_card.layout.display = "none"

    seed.disabled = not use_seed.value

    show_manual = state["need_full"] or state["need_crop"]
    manual_card.layout.display = "flex" if show_manual else "none"
    manual_full.layout.visibility = "visible" if state["need_full"] else "hidden"
    manual_crop.layout.visibility = "visible" if state["need_crop"] else "hidden"

def _on_paths_changed(_=None):
    fp = img_path.value.strip()
    cp = crop_path.value.strip()

    if fp != state["last_full_path"]:
        state["confirmed_full"] = False
        state["last_full_path"] = fp
    if cp != state["last_crop_path"]:
        state["confirmed_crop"] = False
        state["last_crop_path"] = cp

    state["need_full"] = False
    state["need_crop"] = False
    manual_msg.value = ""
    _sync_ui()

img_path.observe(_on_paths_changed, names="value")
crop_path.observe(_on_paths_changed, names="value")
mode.observe(lambda _: (_on_paths_changed(), _sync_ui()), names="value")
use_seed.observe(lambda _: _sync_ui(), names="value")

# ----------------------------
# Main Handler
# ----------------------------
def on_run_clicked(_):
    global img_full, crop_img_proc, PIXEL_SIZE_UM, PIXEL_SIZE_UM_PATCH, CROP_PATH, RESULT_DIR
    global _NUCLEISKY_LAST_MANUAL_FULL, _NUCLEISKY_LAST_MANUAL_CROP

    status_html.value = _spinner_html("Processing...")
    fig_out.clear_output(wait=True)

    try:
        _on_paths_changed()
        full_path = img_path.value.strip()
        if not full_path or not Path(full_path).exists(): raise FileNotFoundError("Full image not found.")

        use_rand = (mode.value == "random")
        if not use_rand:
            cp = crop_path.value.strip()
            if not cp or not Path(cp).exists(): raise FileNotFoundError("Crop image not found.")

        full_raw = load_image(full_path)
        img_full_2d = require_2d_image(full_raw, label="Full image")

        missing = []
        pix_full_meta = get_pixel_size_um_from_tiff(full_path)

        if pix_full_meta is None:
            if state["confirmed_full"]:
                pix_full = require_positive_float(manual_full.value, label="Full µm/px")
                pix_full_src = "manual"
            else:
                missing.append("Full Image")
                state["need_full"] = True
        else:
            pix_full = float(pix_full_meta)
            pix_full_src = "metadata"
            state["need_full"] = False
            state["confirmed_full"] = False

        pix_crop = None
        pix_crop_src = None

        if not use_rand:
            pix_crop_meta = get_pixel_size_um_from_tiff(crop_path.value.strip())
            if pix_crop_meta is None:
                if state["confirmed_crop"]:
                    pix_crop = require_positive_float(manual_crop.value, label="Crop µm/px")
                    pix_crop_src = "manual"
                else:
                    missing.append("Crop Image")
                    state["need_crop"] = True
            else:
                pix_crop = float(pix_crop_meta)
                pix_crop_src = "metadata"
                state["need_crop"] = False
                state["confirmed_crop"] = False

        if missing:
            if state["need_full"]: state["confirmed_full"] = True
            if state["need_crop"]: state["confirmed_crop"] = True
            manual_msg.value = f"<div class='ns-status-error'><b>Pixel size not found:</b> {', '.join(missing)}.<br>Please, provide the required pixel size(s) and click <b>Load & Process</b> again.</div>"
            status_html.value = ""
            _sync_ui()
            return

        out_dir = output_dir.value.strip()
        if out_dir:
            output_dir_msg = f"Using specified output directory: {out_dir}"
            RESULT_DIR = Path(out_dir)
            RESULT_DIR.mkdir(parents=True, exist_ok=True)
        else:
            RESULT_DIR = make_result_dir(big_image_path=full_path, tag="NucleiSky")
            output_dir_msg = f"No output directory specified. Using the auto-generated directory: {RESULT_DIR}"

        if use_rand:
            if zoom_min.value <= 0 or zoom_max.value < zoom_min.value:
                raise ValueError("Invalid zoom range.")
            rng = np.random.default_rng(int(seed.value)) if use_seed.value else None

            crop_2d, crop_pix_size, _ = generate_random_crop(
                img_full_2d, int(patch_h.value), int(patch_w.value),
                (float(zoom_min.value), float(zoom_max.value)),
                float(max_angle.value), float(pix_full), rng=rng
            )
            crop_2d = require_2d_image(crop_2d, label="Generated crop")
            CROP_PATH = str(Path(RESULT_DIR) / "random_crop.tif")
            imwrite(CROP_PATH, np.asarray(crop_2d))
            pix_crop = float(crop_pix_size)
            pix_crop_src = "derived"
        else:
            crop_raw = load_image(crop_path.value.strip())
            crop_2d = require_2d_image(crop_raw, label="Crop image")
            CROP_PATH = crop_path.value.strip()

        _NUCLEISKY_LAST_MANUAL_FULL = float(manual_full.value)
        _NUCLEISKY_LAST_MANUAL_CROP = float(manual_crop.value)
        globals()["_NUCLEISKY_LAST_MANUAL_FULL"] = _NUCLEISKY_LAST_MANUAL_FULL
        globals()["_NUCLEISKY_LAST_MANUAL_CROP"] = _NUCLEISKY_LAST_MANUAL_CROP

        state["need_full"] = False; state["need_crop"] = False
        manual_msg.value = ""
        _sync_ui()

        img_full = img_full_2d
        crop_img_proc = crop_2d
        PIXEL_SIZE_UM = float(pix_full)
        PIXEL_SIZE_UM_PATCH = float(pix_crop)

        save_json(Path(RESULT_DIR) / "inputs_metadata.json", {
            "full_image_path": str(full_path),
            "crop_image_path": str(CROP_PATH),
            "pixel_size_full": PIXEL_SIZE_UM,
            "pixel_size_crop": PIXEL_SIZE_UM_PATCH,
            "mode": mode.value
        })

        status_html.value = _status_ok([
            f"<b>Success!</b>",
            f" {output_dir_msg}",
            f"Full: {PIXEL_SIZE_UM:.4f} µm/px ({pix_full_src})",
            f"Crop: {PIXEL_SIZE_UM_PATCH:.4f} µm/px ({pix_crop_src})"
        ])

        with fig_out:
            fig, axes = plt.subplots(1, 2, figsize=(8, 4), dpi=100)
            imshow_safe(axes[0], img_full_2d, title="Full Image", max_dim=2500)
            imshow_safe(axes[1], crop_2d,     title="Crop (Target)", max_dim=2500)
            plt.tight_layout()
            plt.show()

    except Exception as e:
        status_html.value = _status_err([str(e)], title="Error during processing")

run_button.on_click(on_run_clicked)

# ----------------------------
# Final Display
# ----------------------------
inner = widgets.VBox([
    title_html,
    input_card,
    manual_card,
    output_card,
    run_button,
    widgets.HBox([status_html], layout=widgets.Layout(margin="10px 0")),
    fig_out
])
inner.add_class("ns-root")
ui = widgets.VBox([
    style_html,
    inner
], layout=widgets.Layout(width="90%", margin="0 auto"))

_sync_ui()
display(ui)

### **Segmentation & Feature Extraction**

In this step, the tool identifies individual nuclei ("the stars") to build the geometric constellations used for matching.

**1. Choose Your Strategy**

* **Run New Segmentation:** Use built-in AI or classical algorithms to detect nuclei from scratch.
* **Load Existing Masks:** If you have already segmented your images (e.g., using ImageJ, QuPath, or a separate pipeline), select this to load your TIFF label masks directly.

**2. Segmentation Methods (New Segmentation)**

* **Cellpose:** A standard deep-learning model. Excellent general-purpose accuracy for nuclei but requires a GPU for reasonable speed.
* **InstanSeg:** A highly optimized, blazing-fast deep-learning model. Recommended for very large reference images.
* **Auto-threshold:** Classical computer vision (Otsu, Li, etc.). Extremely fast but requires clean, high-contrast images (e.g., DAPI on a black background).

**3. Advanced Configuration**

* **Model/Method Settings:** You can fine-tune parameters like *minimum object size* (to filter out noise/debris) or specific model versions inside the "Advanced" dropdowns.
* **Scale Normalization:** The pipeline automatically rescales your images to a standard size for segmentation, then maps the coordinates back to the original micron scale.

**Output:**
This step produces `.csv` files containing the centroid coordinates () and shape features for every nucleus found in both images.


In [ ]:
#@title Run to load the segmentation and feature extraction GUI

# ----------------------------
# UI Styling
# ----------------------------

display(widgets.HTML(STYLE_CSS))

# ----------------------------
# WIDGETS
# ----------------------------

# --- Card 1: Source Selection ---
source_mode = widgets.ToggleButtons(
    options=[('Run New Segmentation', 'new'), ('Load Existing Masks', 'existing')],
    value='new',
    description='',
    tooltips=['Run algorithms like Cellpose or InstanSeg', 'Load pre-computed binary TIFF masks'],
    layout=widgets.Layout(width='99%'),
    style={'button_width': 'auto'}
)

# --- Path Inputs (For Existing Masks) ---
mask_path_full = widgets.Text(description="Full Mask", placeholder="/path/to/mask_full.tif", layout=widgets.Layout(width="98%"))
mask_path_crop = widgets.Text(description="Crop Mask", placeholder="/path/to/mask_crop.tif", layout=widgets.Layout(width="98%"))

mask_input_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Mask Paths</div>"),
    mask_path_full,
    widgets.HTML("<div style='height:8px'></div>"),
    mask_path_crop
])
mask_input_card.add_class("ns-card")
mask_input_card.layout.display = "none" # Hidden by default

# --- Card 2: Segmentation Method ---
seg_method = widgets.Dropdown(
    options=[
        ("Cellpose (Deep Learning)", "cellpose"),
        ("InstanSeg (Fast Deep Learning)", "instanseg"),
        ("Auto-threshold (Classic CV)", "threshold"),
    ],
    value="cellpose",
    layout=widgets.Layout(width="98%")
)

method_desc = widgets.HTML("") # Dynamic description

# ---- InstanSeg Specifics ----
inst_model = widgets.Dropdown(options=["brightfield_nuclei", "fluorescence_nuclei_and_cells"], value="brightfield_nuclei", description="Model")
inst_target = widgets.Dropdown(options=["nuclei", "cells"], value="nuclei", description="Target")

# Advanced InstanSeg
inst_mode = widgets.Dropdown(options=[("Auto (small/medium)", "auto"), ("Small image", "small"), ("Medium (tiled)", "medium")], value="auto", description="Mode")
inst_clean = widgets.Checkbox(value=True, description="Cleanup Fragments")
inst_px_ovr = widgets.Checkbox(value=False, description="Override µm/px")
inst_px_val = widgets.FloatText(value=0.5, description="Value", layout=widgets.Layout(width="150px"), disabled=True)
inst_verb = widgets.IntSlider(value=0, min=0, max=2, description="Verbosity")

inst_adv_ui = widgets.VBox([
    inst_mode,
    widgets.HBox([inst_px_ovr, inst_px_val]),
    inst_clean,
    inst_verb
])
inst_adv_ui.add_class("ns-accordion-body")

inst_accord = widgets.Accordion(children=[inst_adv_ui], titles=('Advanced InstanSeg Settings',))
inst_accord.add_class("ns-accordion")

inst_panel = widgets.VBox([
    widgets.HTML("<div class='ns-label'>Model Configuration</div>"),
    widgets.HBox([inst_model, inst_target], layout=widgets.Layout(gap="20px")),
    widgets.HTML("<div style='height:8px'></div>"),
    inst_accord
])

# ---- Threshold Specifics ----
thr_method = widgets.Dropdown(options=["otsu", "li", "yen", "triangle", "isodata"], value="otsu", description="Algorithm")
thr_channel = widgets.IntText(value=0, description="Channel Index", layout=widgets.Layout(width="180px"))

# Advanced Threshold
thr_sigma = widgets.FloatSlider(value=1.0, min=0, max=5, step=0.25, description="Blur Sigma")
thr_min_obj = widgets.IntText(value=80, description="Min Area (px)")
thr_watershed = widgets.Checkbox(value=True, description="Watershed Split")
thr_peak = widgets.IntText(value=5, description="Peak Dist")

thr_adv_ui = widgets.VBox([
    thr_sigma,
    widgets.HBox([thr_min_obj, thr_watershed, thr_peak])
])
thr_adv_ui.add_class("ns-accordion-body")

thr_accord = widgets.Accordion(children=[thr_adv_ui], titles=('Advanced CV Parameters',))
thr_accord.add_class("ns-accordion")

thr_panel = widgets.VBox([
    widgets.HTML("<div class='ns-label'>Thresholding Configuration</div>"),
    widgets.HBox([thr_method, thr_channel], layout=widgets.Layout(gap="20px")),
    widgets.HTML("<div style='height:8px'></div>"),
    thr_accord
])

# Container for all method panels
method_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Segmentation Strategy</div>"),
    seg_method,
    widgets.HTML("<div style='height:12px'></div>"),
    method_desc,
    inst_panel,
    thr_panel
])
method_card.add_class("ns-card")

# --- Card 3: Execution ---
seg_btn = widgets.Button(
    description=" Run Segmentation",
    button_style="primary",
    icon="cogs",
    layout=widgets.Layout(width="99%", height="45px")
)

seg_out = widgets.Output(layout=widgets.Layout(max_height="600px"))
# This redirects the loggin to the output widget
configure_notebook_logging(seg_out)

seg_status = widgets.HTML("")
seg_status.layout.display = "none" # Hidden by default
seg_status.add_class("ns-card")

# ----------------------------
# LOGIC & BINDINGS
# ----------------------------

def _update_ui_state(_=None):
    is_existing = (source_mode.value == 'existing')

    # Toggle Cards
    mask_input_card.layout.display = "flex" if is_existing else "none"
    method_card.layout.display = "none" if is_existing else "flex"

    # Toggle Method Panels
    m = seg_method.value
    inst_panel.layout.display = "flex" if m == "instanseg" else "none"
    thr_panel.layout.display = "flex" if m == "threshold" else "none"

    # Dynamic Description & Image Tags
    if m == "cellpose":
        method_desc.value = "<i>Standard deep learning model for cellular segmentation. Good general purpose choice.</i>"
    elif m == "instanseg":
        method_desc.value = "<i>High-speed pytorch implementation. Best for large datasets.</i>"
    elif m == "threshold":
        method_desc.value = "<i>Classical computer vision. Fast, but requires high contrast and clean images.</i>"
    method_desc.value = f"<div class='ns-page-subtitle'>{method_desc.value}</div>"
    # Button Text
    seg_btn.description = " Load Masks & Extract Features" if is_existing else " Run Segmentation & Extraction"

# InstanSeg Override logic
def _toggle_inst_px(_=None):
    inst_px_val.disabled = not inst_px_ovr.value
inst_px_ovr.observe(_toggle_inst_px, names="value")

source_mode.observe(_update_ui_state, names="value")
seg_method.observe(_update_ui_state, names="value")

# Initialize
_update_ui_state()

# ----------------------------
# EXECUTION HANDLER
# ----------------------------
def on_segment_click(_):
    # Mapping global variables to new UI
    global df_full, df_crop, masks_full, masks_crop
    global img_full_seg, crop_img_seg
    global PIXEL_SIZE_UM_FULL_SEG, PIXEL_SIZE_UM_CROP_SEG
    global RESCALE_FACTOR_FULL, RESCALE_FACTOR_CROP, TARGET_UM_PER_PX

    seg_out.clear_output(wait=True)

    with seg_out:
        try:
            # 1. Pre-flight Checks
            required = ("img_full", "crop_img_proc", "PIXEL_SIZE_UM", "PIXEL_SIZE_UM_PATCH")
            missing = [k for k in required if k not in globals()]
            if missing: raise RuntimeError(f"Missing Step 1 globals: {missing}")

            # 2. Get Data
            img_full_orig = require_2d_image(np.asarray(img_full), label="Full Image")
            img_crop_orig = require_2d_image(np.asarray(crop_img_proc), label="Crop Image")
            pix_full_orig = float(globals()["PIXEL_SIZE_UM"])
            pix_crop_orig = float(globals()["PIXEL_SIZE_UM_PATCH"])

            # 3. Setup Directories

            # NOTE: Avoid eager evaluation of make_result_dir (it has side effects: creates a folder).
            if "RESULT_DIR" in globals():
                RESULT_DIR_LOCAL = Path(globals()["RESULT_DIR"])
            else:
                RESULT_DIR_LOCAL = Path(make_result_dir(tag="NucleiSky"))

            SEG_DIR = RESULT_DIR_LOCAL / "segmentation"
            SEG_DIR.mkdir(parents=True, exist_ok=True)

            # 4. Processing Mode Logic
            use_masks = (source_mode.value == 'existing')

            if use_masks:
                # BYPASS SCALING
                img_full_seg, crop_img_seg = img_full_orig, img_crop_orig
                PIXEL_SIZE_UM_FULL_SEG, PIXEL_SIZE_UM_CROP_SEG = pix_full_orig, pix_crop_orig
                RESCALE_FACTOR_FULL, RESCALE_FACTOR_CROP = 1.0, 1.0
                print(f"📂 Loading existing masks...")

                if not mask_path_full.value.strip(): raise ValueError("Full mask path required")
                if not mask_path_crop.value.strip(): raise ValueError("Crop mask path required")

                masks_full = require_2d_label_mask(load_image(mask_path_full.value.strip()), label="Full Mask", expected_shape=img_full_orig.shape)
                masks_crop = require_2d_label_mask(load_image(mask_path_crop.value.strip()), label="Crop Mask", expected_shape=img_crop_orig.shape)

            else:
                # PERFORM SCALING & SEGMENTATION
                print("⚖️ Normalizing scale for AI/CV models...")
                (img_full_seg, crop_img_seg, PIXEL_SIZE_UM_FULL_SEG, PIXEL_SIZE_UM_CROP_SEG,
                 RESCALE_FACTOR_FULL, RESCALE_FACTOR_CROP, TARGET_UM_PER_PX) = scale_normalize_pair_for_segmentation(
                    img_full=img_full_orig, img_crop=img_crop_orig,
                    pixel_size_full_um=pix_full_orig, pixel_size_crop_um=pix_crop_orig,
                    strategy="coarsest", max_upsample=10.0, min_downsample=0.1, dtype_out=np.float32
                )

                # Build Settings Dictionary
                settings = {"method": seg_method.value}

                # Override logic
                pix_dispatch_full = float(PIXEL_SIZE_UM_FULL_SEG)
                pix_dispatch_crop = float(PIXEL_SIZE_UM_CROP_SEG)

                if seg_method.value == "instanseg":
                    if inst_px_ovr.value:
                        pix_dispatch_full = pix_dispatch_crop = float(inst_px_val.value)

                    settings["instanseg"] = {
                        "model_name": inst_model.value, "target": inst_target.value,
                        "mode": inst_mode.value, "verbosity": inst_verb.value,
                        "pixel_size_um": float(pix_dispatch_full),
                        "cleanup_fragments": inst_clean.value
                    }
                elif seg_method.value == "threshold":
                    settings["threshold"] = {
                        "threshold_method": thr_method.value, "channel": thr_channel.value,
                        "gaussian_sigma": thr_sigma.value, "min_object_size": thr_min_obj.value,
                        "do_watershed": thr_watershed.value, "peak_min_distance": thr_peak.value
                    }

                print(f"🧠 Running {seg_method.value}...")
                seg_status.layout.display = "flex"
                seg_status.value = _spinner_html("Segmentation in progress...")
                masks_full = segment_nuclei_dispatch(img_full_seg, seg_method.value, pix_dispatch_full, settings)
                masks_crop = segment_nuclei_dispatch(crop_img_seg, seg_method.value, pix_dispatch_crop, settings)
                seg_status.value = _status_html("Segmentation complete.", kind='ok')

            # 5. Feature Extraction (Common)
            print("📊 Extracting morphology features...")
            df_full = extract_nuclear_features(masks_full, None, float(PIXEL_SIZE_UM_FULL_SEG), k_neighbors=10)
            df_crop = extract_nuclear_features(masks_crop, None, float(PIXEL_SIZE_UM_CROP_SEG), k_neighbors=10)

            # Map back to original scale
            df_full = add_centroids_orig_px_columns(df_full, float(RESCALE_FACTOR_FULL))
            df_crop = add_centroids_orig_px_columns(df_crop, float(RESCALE_FACTOR_CROP))

            # 6. Visualization
            print(f"✅ Extracted: Full={len(df_full)} nuclei, Crop={len(df_crop)} nuclei")

            # Save CSVs
            df_full.to_csv(SEG_DIR / "features_full.csv", index=False)
            df_crop.to_csv(SEG_DIR / "features_crop.csv", index=False)

            imwrite(
                SEG_DIR / "mask_full.tif",
                masks_full.astype(np.int32),
                compression="zlib"
            )
            imwrite(
                SEG_DIR / "mask_crop.tif",
                masks_crop.astype(np.int32),
                compression="zlib"
            )

            # Define the threshold for an image to be too big
            big_image_size = 5000
            # Define the zoom patch size
            zoom_patch_size = 1000

            # Plot using safe downsampling for images and points
            fig, ax = plt.subplots(2, 2, figsize=(10, 10))

            # --- Safe Visualization: Full Image ---
            disp_full = imshow_safe(ax[0, 0], img_full_orig, title=f"Full ({len(df_full)})")
            # Calculate coordinate scale (Display Shape / Original Shape)
            scale_y_f, scale_x_f = disp_full.shape[0] / img_full_orig.shape[0], disp_full.shape[1] / img_full_orig.shape[1]

            cent_f = centroids_from_df(df_full, use_um=False, use_orig_px=True)
            if len(cent_f):
                cent_f_safe = downsample_points_for_display(cent_f)
                # Apply scale to align points with the downsampled image axes
                ax[0, 0].scatter(cent_f_safe[:,1] * scale_x_f, cent_f_safe[:,0] * scale_y_f, s=1, c='r', alpha=0.5)

            # Check if the full image is too large
            if max(img_full_orig.shape) > big_image_size:
                # If so, take a random patch and display it in the second subplot
                random_x = np.random.randint(0, img_full_orig.shape[1] - zoom_patch_size)
                random_y = np.random.randint(0, img_full_orig.shape[0] - zoom_patch_size)
                disp_full_patch = imshow_safe(ax[1, 0],
                                              img_full_orig[random_y:random_y+zoom_patch_size, random_x:random_x+zoom_patch_size],
                                              title="Full Image Patch")

                scale_y_pf, scale_x_pf = disp_full_patch.shape[0] / zoom_patch_size, disp_full_patch.shape[1] / zoom_patch_size
                if len(cent_f):
                    cent_f_patch = cent_f[(cent_f[:,0] >= random_y) & (cent_f[:,0] < random_y + zoom_patch_size) &
                                            (cent_f[:,1] >= random_x) & (cent_f[:,1] < random_x + zoom_patch_size)]
                    cent_f_patch_safe = downsample_points_for_display(cent_f_patch)
                    ax[1, 0].scatter((cent_f_patch_safe[:,1] - random_x) * scale_x_pf, (cent_f_patch_safe[:,0] - random_y) * scale_y_pf, s=5, c='r', alpha=0.5)
            else:
                ax[1, 0].axis('off')  # Hide the second subplot if not used

            # --- Safe Visualization: Crop Image ---
            disp_crop = imshow_safe(ax[0, 1], img_crop_orig, title=f"Crop ({len(df_crop)})")
            # Calculate coordinate scale (Display Shape / Original Shape)
            scale_y_c, scale_x_c = disp_crop.shape[0] / img_crop_orig.shape[0], disp_crop.shape[1] / img_crop_orig.shape[1]

            cent_c = centroids_from_df(df_crop, use_um=False, use_orig_px=True)
            if len(cent_c):
                cent_c_safe = downsample_points_for_display(cent_c)
                # Apply scale to align points with the downsampled image axes
                ax[0, 1].scatter(cent_c_safe[:,1] * scale_x_c, cent_c_safe[:,0] * scale_y_c, s=1, c='r', alpha=0.5)

            # Check if the crop image is too large
            if max(img_crop_orig.shape) > big_image_size  :
                # If so, take a random patch and display it in the second subplot
                random_x = np.random.randint(0, img_crop_orig.shape[1] - zoom_patch_size)
                random_y = np.random.randint(0, img_crop_orig.shape[0] - zoom_patch_size)
                disp_crop_patch = imshow_safe(ax[1, 1],
                                              img_crop_orig[random_y:random_y+zoom_patch_size, random_x:random_x+zoom_patch_size],
                                              title="Crop Image Patch")

                scale_y_pc, scale_x_pc = disp_crop_patch.shape[0] / zoom_patch_size, disp_crop_patch.shape[1] / zoom_patch_size
                if len(cent_c):
                    cent_c_patch = cent_c[(cent_c[:,0] >= random_y) & (cent_c[:,0] < random_y + zoom_patch_size) &
                                            (cent_c[:,1] >= random_x) & (cent_c[:,1] < random_x + zoom_patch_size)]
                    cent_c_patch_safe = downsample_points_for_display(cent_c_patch)
                    ax[1, 1].scatter((cent_c_patch_safe[:,1] - random_x) * scale_x_pc, (cent_c_patch_safe[:,0] - random_y) * scale_y_pc, s=5, c='r', alpha=0.5)
            else:
                ax[1, 1].axis('off')  # Hide the second subplot if not used

            plt.tight_layout()
            plt.show()

        except Exception as e:
            print("❌ Error:")
            traceback.print_exc()

seg_btn.on_click(on_segment_click)

# ----------------------------
# UI Layout
# ----------------------------

inner = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Choose to apply segmentation or load existing masks</div>"),
    source_mode,
    widgets.HTML("<div style='height:16px'></div>"),
    mask_input_card,
    method_card,
    widgets.HTML("<div style='height:16px'></div>"),
    seg_btn,
    seg_status,
    seg_out
])
inner.add_class("ns-root")
ui = widgets.VBox([
    style_html,
    inner
], layout=widgets.Layout(width="90%", margin="0 auto"))

display(ui)

# Step 2: NucleiSky matching
---

### **Option A: Run NucleiSky2D Matching**

This is the core step where the "constellation" of nuclei in your crop is located within the reference image.

**1. How it Works**
The pipeline executes **four different pattern matching algorithms** to maximize the chance of finding the correct location, even if segmentation is imperfect:

* **Graph:** Matches the topological structure (connections) between neighbors.
* **Quad & Triangles:** Matches specific geometric shapes formed by nuclei groups.
* **Geometric Hashing:** A voting system based on invariant coordinates.

**2. Configuration**

* **Use Default Settings (Recommended):** Works for most standard microscopy images.
* **Advanced Customization:** Uncheck the box to reveal tabs where you can tweak specific parameters (e.g., scale search range, error tolerances) if the default matching fails.

**3. Outputs**

* **Visual QC:** The cell will display plots showing the alignment (Green = Reference, Magenta = Aligned Crop).
* **Exports:** Aligned images and transformation data (`.json`) are automatically saved to your results folder.

In [ ]:
#@title Run NucleiSky2D
# ============================================

# ----------------------------
# UI Styling
# ----------------------------

display(widgets.HTML(STYLE_CSS))

# ----------------------------
# Config Widget Builders
# ----------------------------

def _make_value_widget(key: str, val):
    """Creates a styled input widget based on value type."""
    layout = widgets.Layout(width="99%")

    if isinstance(val, bool):
        return widgets.Checkbox(value=val, description="", indent=False, layout=layout)
    if isinstance(val, int) and not isinstance(val, bool):
        return widgets.IntText(value=val, layout=layout)
    if isinstance(val, float):
        return widgets.FloatText(value=val, layout=layout)

    return widgets.Text(value=("None" if val is None else str(val)), layout=layout)

def _parse_text_value(text_val):
    s = str(text_val).strip()
    if s == "None": return None
    if s.lower() == "true": return True
    if s.lower() == "false": return False
    try:
        f = float(s)
        return int(f) if f.is_integer() else f
    except ValueError:
        return s

def create_matcher_config_widget(cfg_init: dict):
    """Builds the configuration tabs with get/set hooks."""
    cfg_init = copy.deepcopy(cfg_init)
    sections = ["_common", "graph", "quad", "triangles", "hashing"]

    section_meta = {
        "_common": (
            "General",
            "Global thresholds for all matchers."
        ),
        "graph": (
            "Graph Match",
            "Topological matching using Delaunay triangulation edges."
        ),
        "quad": (
            "Quad",
            "Geometric hashing using 4-point constellations."
        ),
        "triangles": (
            "Triangles",
            "Geometric hashing using 3-point constellations."
        ),
        "hashing": (
            "Geo. Hashing",
            "Invariant voting system."
        ),
    }

    widget_map = {}
    tabs = []
    titles = []

    for sec in sections:
        if sec not in cfg_init:
            continue

        widget_map[sec] = {}
        rows = []

        meta = section_meta.get(sec, (sec, ""))

        titles.append(meta[0])

        desc = widgets.HTML(
            f"<div class='ns-desc'>{meta[1]}</div>"
        )
        rows.append(desc)

        items = cfg_init.get(sec, {})

        for k in sorted(items.keys()):
            w = _make_value_widget(k, items[k])
            w.layout = widgets.Layout(
                flex="1 1 auto",
                width="auto"
            )

            widget_map[sec][k] = w

            label = widgets.HTML(
                f"<div class='ns-label'>{k}</div>",
                layout=widgets.Layout(
                    width="180px",
                    min_width="180px"
                )
            )

            row = widgets.HBox(
                [label, w],
                layout=widgets.Layout(
                    width="99%",
                    margin="0",
                    align_items="center"
                )
            )
            row.add_class("ns-tab-row")

            rows.append(row)

        tab_body = widgets.VBox(
            rows,
            layout=widgets.Layout(
                padding="0",
                width="99%"
            )
        )
        tab_body.add_class("ns-tab-panel")

        tabs.append(tab_body)

    ui_tabs = widgets.Tab(children=tabs)
    ui_tabs.add_class("ns-tabs")

    for i, t in enumerate(titles):
        ui_tabs.set_title(i, t)

    def get_config():
        cfg = {sec: {} for sec in sections if sec in widget_map}

        for sec in cfg:
            for k, w in widget_map[sec].items():
                if isinstance(w, widgets.Text):
                    cfg[sec][k] = _parse_text_value(w.value)
                else:
                    cfg[sec][k] = w.value

        return cfg

    def set_config(cfg_new: dict):
        cfg_new = copy.deepcopy(cfg_new)

        for sec in widget_map:
            items = cfg_new.get(sec, {})

            for k, w in widget_map[sec].items():
                if k in items:
                    v = items[k]

                    if isinstance(w, widgets.Text):
                        w.value = "None" if v is None else str(v)
                    else:
                        w.value = v

    return ui_tabs, get_config, set_config

# ----------------------------
# Main UI Construction
# ----------------------------

# 1. Info Card
info_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Pipeline Overview</div>"),
    widgets.HTML("""
      <div class='ns-body-text'>
      This step executes <b>four pattern matching algorithms</b> one after the other to find the crop location:
      <ul>
        <li><b>Graph</b>: Matches topology</li>
        <li><b>Quad & Triangles</b>: Matches geometric constellations</li>
        <li><b>Hashing</b>: Voting based on invariant coordinates</li>
      </ul>
      </div>
    """)
])
info_card.add_class("ns-card")

# 2. Config Card
MATCHER_UI_TABS, get_cfg, set_cfg = create_matcher_config_widget(DEFAULT_MATCHER_CONFIG)

use_default_box = widgets.Checkbox(value=True, description="Use default settings", indent=False)
customize_box = widgets.VBox([
    widgets.HTML("<div style='height:8px'></div>"),
    MATCHER_UI_TABS
])
customize_box.layout.display = "none" # Hidden by default

def _toggle_custom(_):
    customize_box.layout.display = "none" if use_default_box.value else "block"

use_default_box.observe(_toggle_custom, names="value")

config_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Configuration</div>"),
    use_default_box,
    customize_box
])
config_card.add_class("ns-card")

# 3. Execution
run_btn = widgets.Button(
    description=" Run NucleiSky2D",
    button_style="success",
    icon="bolt",
    layout=widgets.Layout(width="99%", height="45px", font_weight="bold")
)
ui_out = widgets.Output(layout=widgets.Layout(max_height="600px", overflow="auto"))

matcher_status = widgets.HTML(value="")
matcher_status.layout.display = "none"
matcher_status.add_class("ns-card")

# ----------------------------
# Logic (Runner)
# ----------------------------
def _run_matching_pipeline(cfg_selected: dict):
    # --- Validation ---
    img_full_local = require_2d_image(globals()["img_full"], label="img_full (original)")
    crop_img_proc_local = require_2d_image(globals()["crop_img_proc"], label="crop_img_proc (original)")

    if len(df_full) == 0 or len(df_crop) == 0:
        raise RuntimeError("Features missing. Run Step 2 first.")

    # --- Data Prep ---
    centroids_full_um = extract_centroids_um(df_full, name="df_full")
    centroids_crop_um = extract_centroids_um(df_crop, name="df_crop")
    features_full = stack_feature_vectors(df_full, name="df_full")
    features_crop = stack_feature_vectors(df_crop, name="df_crop")

    # Image Refs (ORIGINAL ONLY)
    img_full_match = img_full_local
    img_crop_match = crop_img_proc_local
    pix_full_match = require_positive_float(float(PIXEL_SIZE_UM), label="pix_full")
    pix_crop_match = require_positive_float(float(PIXEL_SIZE_UM_PATCH), label="pix_crop")

    print(f"📐 Matching context: Full={img_full_match.shape}, Crop={img_crop_match.shape}")

    base_save = Path(RESULT_DIR) / "matching"
    base_save.mkdir(parents=True, exist_ok=True)

    # --- Execution Helper ---
    def _run_one(matcher_name: str, **kwargs):
        print(f"⚙️ Running {matcher_name} matcher...")
        matcher_status.layout.display = "flex"
        matcher_status.value = _spinner_html("Matching in progress...")
        try:
            res = NucleiSky(
                centroids_crop_um=centroids_crop_um,
                centroids_full_um=centroids_full_um,
                img_full=img_full_match,
                img_crop=img_crop_match,
                ij_percentile_normalize=ij_percentile_normalize,
                pixel_size_full_um=pix_full_match,
                pixel_size_crop_um=pix_crop_match,
                matcher=matcher_name,
                df_full=df_full,
                df_crop=df_crop,
                matcher_config=cfg_selected,
                **kwargs,
            )
            matcher_status.value = _status_html("Matching  complete.", kind='ok')
            return res
        except Exception:
            matcher_status.value = _status_html("Matching failed.", kind='err')
            print(f"❌ {matcher_name} crashed.")
            traceback.print_exc()
            return dict(success=False, matcher=matcher_name)

    def _maybe_show(res, tag):
        if not res.get("success", False):
            return
        try:
            show_alignment_original_and_rescaled(
                res,
                save_dir=base_save / tag,
                img_full_orig=img_full_local,
                img_crop_orig=crop_img_proc_local,
                pixel_size_full_orig_um=float(PIXEL_SIZE_UM),
                pixel_size_crop_orig_um=float(PIXEL_SIZE_UM_PATCH),
            )
        except Exception:
            traceback.print_exc()

    # --- Run Matchers ---
    results_map = {}

    res_graph = _run_one("graph", features_crop=features_crop, features_full=features_full)
    _maybe_show(res_graph, "graph")
    results_map["graph"] = res_graph

    for m in ["quad", "triangles", "hashing"]:
        res = _run_one(m)
        _maybe_show(res, m)
        results_map[m] = res

    # --- Export (ROI ONLY) ---
    print("\n💾 Exporting results (ROI only)...")
    out_dir = base_save / "exports_nucleisky"
    out_dir.mkdir(parents=True, exist_ok=True)
    jsonl_path = out_dir / "transforms.jsonl"

    # Optional: avoid duplicate appends across reruns
    # if jsonl_path.exists():
    #     jsonl_path.unlink()

    save_matcher_config_json(cfg_selected, out_dir / "matcher_config.json")

    for name, res in results_map.items():
        if not res.get("success") or res.get("best_scale") is None:
            continue

        # 1) Save transform record
        rec = save_nucleisky_transform(
            res,
            out_path=out_dir / f"{name}_transform_original.json",
            matcher_name=name,
            pixel_size_full_um=float(PIXEL_SIZE_UM),
            pixel_size_crop_um=float(PIXEL_SIZE_UM_PATCH),
            require_success=True,
        )
        append_transform_jsonl(rec, jsonl_path)

        # 2) Export ROI stack only (aligned_on_full_px.tif but ROI-sized)
        out_imgs = out_dir / f"{name}_images_original_roi"
        out_imgs.mkdir(parents=True, exist_ok=True)

        ret = export_aligned_imagej_stacks(
            res,
            out_dir=out_imgs,
            img_full=img_full_local,
            img_crop=crop_img_proc_local,
            pixel_size_full_um=float(PIXEL_SIZE_UM),
            pixel_size_crop_um=float(PIXEL_SIZE_UM_PATCH),
            axes_full="YX",
            axes_crop="YX",
            export_region="roi",   # <-- ROI ONLY
            margin_px=20,
            always_two_stacks=False,
            order_intensity=1,
            format="tiff",         # ROI TIFF is typically safe; change if you prefer zarr
        )

        if isinstance(ret, dict) and ret.get("skipped"):
            print(f"  ⚠️ {name}: export skipped ({ret['skipped']})")
        else:
            print(
                f"  ✅ {name.capitalize()}: ROI exported "
                f"(Rot: {rec.get('rotation_deg',0):.1f}°, Scale: {rec.get('scale',1):.2f})"
            )

    # Globals update
    globals()["res_graph"] = results_map["graph"]
    globals()["res_quad"]  = results_map["quad"]
    globals()["res_tri"]   = results_map["triangles"]
    globals()["res_hash"]  = results_map["hashing"]

    return results_map


def _on_run_clicked(_):
    run_btn.disabled = True
    ui_out.clear_output()
    with ui_out:
        try:
            cfg = DEFAULT_MATCHER_CONFIG if use_default_box.value else get_cfg()
            _run_matching_pipeline(cfg)
        except Exception as e:
            traceback.print_exc()
        finally:
            run_btn.disabled = False

run_btn.on_click(_on_run_clicked)

# ----------------------------
# Final Display
# ----------------------------
inner = widgets.VBox([
    widgets.HTML("<div style='height:10px'></div>"),
    info_card,
    config_card,
    run_btn,
    matcher_status,
    ui_out
])
inner.add_class("ns-root")
ui = widgets.VBox([
    style_html,
    inner
], layout=widgets.Layout(width="90%", margin="0 auto"))

display(ui)

### **Option B: Run Adaptive Matching**

This "autopilot" mode automatically attempts multiple matching strategies to locate your crop within the reference image. It is the recommended starting point for most datasets.

* **Strategy:** The tool runs algorithms (like Graph, Quad, or Hashing) in a sequence optimized for your data's nuclei density. It stops as soon as a high-confidence match is found.
* **Settings:**
* **Random Seed:** Ensures the result is reproducible (useful if the matcher uses random sampling).
* **Advanced Constraints:** You can set a **Time Limit** to prevent long searches or adjust the **Export Margin** to add padding around the saved output.


* **Outcome:** The best valid transformation is automatically applied, visualized, and saved to the `adaptive/` results folder.

In [ ]:
#@title Run Adaptive 2D NucleiSky
# ============================================

# ----------------------------
# UI Styling
# ----------------------------

display(widgets.HTML(STYLE_CSS))

# ----------------------------
# UI Components
# ----------------------------

# 1. Header Card
header_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Adaptive Matching Strategy</div>"),
    widgets.HTML(
        "<div class='ns-body-text'>"
        "This step runs a <b>sequential cascade</b> of matchers. It tries algorithms one by one using default settings until a valid transformation is found. "
        "The order of matchers is automatically determined by the number of nuclei in the crop to maximize speed and success rate."
        "</div>"
    )
])
header_card.add_class("ns-card")

# 2. Settings Card
w_base_seed = widgets.IntText(value=0, layout=widgets.Layout(width="200px"))

# Advanced Constraints
w_margin = widgets.IntText(value=20,
                           description="Export Margin (px):",
                           style={'description_width': 'initial'},
                           layout=widgets.Layout(width="200px"))
w_time_limit_chk = widgets.Checkbox(value=False,
                                    description="Enable Time Limit:",
                                    indent=False)
w_time_limit_val = widgets.IntText(value=60,
                                   description="Max Seconds:",
                                   disabled=True,
                                   style={'description_width': 'initial'},
                                   layout=widgets.Layout(width="200px"))

def _toggle_time(_): w_time_limit_val.disabled = not w_time_limit_chk.value
w_time_limit_chk.observe(_toggle_time, names="value")

adv_ui = widgets.VBox([
    widgets.HBox([w_margin]),
    widgets.HBox([w_time_limit_chk, w_time_limit_val])
])

settings_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Configuration</div>"),
    widgets.HBox([widgets.Label("Base Random Seed:", layout=widgets.Layout(width="120px")), w_base_seed]),
    widgets.HTML("<div style='height:8px'></div>"),
    widgets.Accordion(children=[adv_ui], titles=('Advanced Constraints',))
])
settings_card.add_class("ns-card")

# 3. Execution
run_btn = widgets.Button(
    description=" Run adaptive NucleiSky2D",
    button_style="primary",
    icon="rocket",
    layout=widgets.Layout(width="99%", height="45px", font_weight="bold")
)

# Status log widget
status_log = widgets.HTML(value="")
status_log.layout.display = "none"
status_log.add_class("ns-card")

out = widgets.Output(layout=widgets.Layout(max_height="600px", overflow="auto"))

# ----------------------------
# Logic
# ----------------------------
def on_run_adaptive(_):
    run_btn.disabled = True
    status_log.layout.display = "flex"
    status_log.value = _spinner_html("Adaptive search in progress...")
    out.clear_output()

    try:
        # 1. Setup Globals
        base_seed = int(w_base_seed.value)
        margin = int(w_margin.value)
        max_time = int(w_time_limit_val.value) if w_time_limit_chk.value else None

        RESULT_DIR_ADAPTIVE = Path(RESULT_DIR) / "adaptive"
        RESULT_DIR_ADAPTIVE.mkdir(parents=True, exist_ok=True)

        # 2. Run Pipeline
        with out:
            best_out, history = run_adaptive_matching_and_export(
                df_full=df_full,
                df_crop=df_crop,
                img_full=img_full,
                img_crop=crop_img_proc,
                pixel_size_full_um=float(PIXEL_SIZE_UM),
                pixel_size_crop_um=float(PIXEL_SIZE_UM_PATCH),
                result_dir=str(RESULT_DIR_ADAPTIVE),
                img_full_seg=globals().get("img_full_seg", None),
                img_crop_seg=globals().get("crop_img_seg", None),
                pixel_size_full_seg_um=globals().get("PIXEL_SIZE_UM_FULL_SEG", None),
                pixel_size_crop_seg_um=globals().get("PIXEL_SIZE_UM_CROP_SEG", None),
                cfg_selected=DEFAULT_MATCHER_CONFIG,
                base_seed=base_seed,
                margin_px=margin,
                store_full_out=False,
                max_total_time_s=max_time,
                ij_percentile_normalize=ij_percentile_normalize,
            )

        # 3. Success UI Update
        if best_out and best_out.get('success'):
            score = best_out.get('match_quality', {}).get('frac_inliers', 0)
            status_log.value = (
                f"<div style='background:var(--ns-ok-bg); color:var(--ns-ok-text); padding:10px; border-radius:6px; border:1px solid var(--ns-ok-border);'>"
                f"✅ <b>Success!</b> Adaptive search complete.<br>"
                f"Best Inlier Score: <b>{score:.2f}</b><br>"
                f"Results saved to: <code>{RESULT_DIR_ADAPTIVE.name}</code>"
                f"</div>"
            )

            # 4. In case it was a success plot the visualization
            print("\n🎨 Generating alignment visualization...")
            show_alignment_original_and_rescaled(
                best_out,
                img_full_orig=img_full,
                img_crop_orig=crop_img_proc,
                pixel_size_full_orig_um=float(PIXEL_SIZE_UM),
                pixel_size_crop_orig_um=float(PIXEL_SIZE_UM_PATCH),
                save_dir=str(RESULT_DIR_ADAPTIVE),
            )
        else:
            status_log.value = (
                "<div style='background:var(--ns-err-bg); color:var(--ns-err-text); padding:10px; border-radius:6px; border:1px solid var(--ns-err-border);'>"
                "⚠️ Search completed, but no confident match was found. Try increasing time limit or checking segmentation quality."
                "</div>"
            )

    except Exception as e:
        status_log.value = f"<div style='color:var(--ns-err-text);'>❌ Error: {str(e)}</div>"
        with out:
            import traceback
            traceback.print_exc()
    finally:
        run_btn.disabled = False

run_btn.on_click(on_run_adaptive)

# ----------------------------
# Layout
# ----------------------------
inner = widgets.VBox([
    header_card,
    settings_card,
    run_btn,
    status_log,
    out
])
inner.add_class("ns-root")
ui = widgets.VBox([
    style_html,
    inner
], layout=widgets.Layout(width="90%", margin="0 auto"))

display(ui)

# Step 3: Apply saved NucleiSky transform
---

Once you have successfully matched your crop (e.g., using the DAPI channel), you can reuse that transformation to align other channels (e.g., GFP, RFP) or other images from the same dataset without re-running the matching process.

**1. Inputs**

* **Transform:** Path to the `.jsonl` file generated in the previous step (e.g., `adaptive_best_transform.jsonl`).
* **Large Img (Reference):** The target image you want to align *to*.
* **Crop Img (Moving):** The new image you want to align (e.g., a different channel of the same crop).

**2. Review Metadata**

* **Axes:** Ensure the axis labels (e.g., `YX`, `CYX`, `ZCYX`) match your input images. The tool attempts to guess them from the file header, but you should verify them manually.
* **Pixel Size:** Just like in matching, correct physical units () are critical for correct scaling.

**3. Export Settings**
You can choose *how* the aligned image is saved. The most common options are:

* **Reference Grid / ROI:** Saves the crop resampled onto the reference pixel grid, but cropped to the region of interest. (Most common for overlays).
* **Reference Grid / Full Canvas:** Reconstructs the entire reference image size with the crop placed in its correct location. **Warning:** This creates very large files (mostly empty black space). Use `OME-Zarr` format if the reference image is massive.
* **Format:** Choose `TIFF` for standard compatibility or `OME-Zarr` for high-performance handling of large multi-dimensional datasets.

In [ ]:
#@title Apply saved NucleiSky transform

# ============================================
# Constants
# ============================================

PIX_EQUAL_RTOL = 1e-3

# ============================================
# Small UI / HTML helpers
# ============================================


def _get_icon(grid_type, extent_type, active=True):
    """Small placeholder icon for the export option cards."""
    cls = "ns-step-active" if active else "ns-step-idle"

    rect = (
        '<rect x="10" y="10" width="30" height="30" '
        'fill="none" stroke="currentColor" stroke-width="2"/>'
    )

    return f"""
    <svg width="40" height="40" viewBox="0 0 50 50"
         class="{cls}"
         style="border-radius:4px; margin-right:8px; flex-shrink:0; display:inline-block;">
        {rect}
    </svg>
    """


def _desc_width(*widgets_to_style, w="92px"):
    """Apply a common description width to widgets."""
    for widget in widgets_to_style:
        try:
            widget.style.description_width = w
        except Exception:
            pass


def _set_visible(widget, visible: bool, display_mode="flex"):
    """Show/hide a widget using layout.display."""
    widget.layout.display = display_mode if visible else "none"


def _make_text(description, placeholder="", value="", width="98%"):
    return widgets.Text(
        value=value,
        description=description,
        placeholder=placeholder,
        layout=widgets.Layout(width=width),
    )


def _make_button(description, style="", icon="", width="99%", height=None, css_class=None):
    layout_kwargs = {"width": width}

    if height is not None:
        layout_kwargs["height"] = height

    button = widgets.Button(
        description=description,
        button_style=style,
        icon=icon,
        layout=widgets.Layout(**layout_kwargs),
    )

    if css_class:
        button.add_class(css_class)

    return button


def _make_card(children, visible=True):
    card = widgets.VBox(children)
    card.add_class("ns-card")
    card.layout.display = "flex" if visible else "none"
    return card


def _make_section_header(text):
    return widgets.HTML(f"<div class='ns-header'>{html.escape(text)}</div>")


def _make_hr():
    return widgets.HTML(
        "<hr style='border:0; border-top:1px solid var(--ns-border-soft); margin:12px 0;'>"
    )


# ============================================
# Data / validation helpers
# ============================================

def _is_finite_number(x):
    try:
        return math.isfinite(float(x))
    except Exception:
        return False


def _pix_sizes_same(pr, pm, tol=PIX_EQUAL_RTOL):
    if not (_is_finite_number(pr) and _is_finite_number(pm)):
        return False

    pr = float(pr)
    pm = float(pm)

    if pr <= 0 or pm <= 0:
        return False

    mean_pix = 0.5 * (pr + pm)

    return (abs(pr - pm) / max(mean_pix, 1e-12)) < float(tol)


def _guess_axes_from_shape(shape):
    if shape is None:
        return "", ["YX", "CYX", "ZYX", "ZCYX", "TCYX", "TZCYX"]

    nd = len(tuple(shape))

    if nd == 2:
        return "YX", ["YX"]

    if nd == 3:
        return "CYX", ["CYX", "ZYX"]

    if nd == 4:
        return "ZCYX", ["ZCYX", "TZYX", "TCYX"]

    if nd == 5:
        return "TZCYX", ["TZCYX"]

    return "", ["YX", "CYX", "ZCYX", "TZCYX"]


def _preflight_missing():
    required = [
        "load_transforms_any",
        "inspect_image_header",
        "pick_best_transform",
        "run_export_with_record",
    ]

    return [name for name in required if name not in globals()]


def _format_option(rec: dict):
    mq = rec.get("match_quality", {}) or {}

    frac = mq.get("frac_inliers", None)
    err = mq.get("mean_error_um", None)

    frac_s = f"{float(frac):.2f}" if frac is not None and _is_finite_number(frac) else "?"
    err_s = f"{float(err):.3g}" if err is not None and _is_finite_number(err) else "?"

    return (
        f"{rec.get('matcher', '?')} | "
        f"line={rec.get('_line', 0)} | "
        f"inliers={frac_s} | "
        f"err={err_s}um"
    )


def _set_axes_widget_from_header(widget, header):
    axes = header.get("axes") or ""

    if axes:
        widget.options = [axes]
        widget.value = axes
        return

    _, candidates = _guess_axes_from_shape(header.get("shape"))

    widget.options = candidates
    widget.value = candidates[0] if candidates else ""


def _default_output_dir():
    try:
        if "RESULT_DIR" in globals():
            return str(Path(globals()["RESULT_DIR"]) / "aligned_from_transform")
    except Exception:
        pass

    return ""


# ============================================
# State
# ============================================

STATE = {
    "inspected": False,
    "candidates": [],
    "choice_to_record": {},
    "selected": None,
    "ref_header": None,
    "crop_header": None,
    "pix_same": None,
}


# ============================================
# Widgets
# ============================================

style_html = widgets.HTML(STYLE_CSS)

title = widgets.HTML(
    """
    <div class='ns-page-title'>Apply Saved NucleiSky Transform</div>
    <div class='ns-page-subtitle'>Select a transform record and apply it to new images.</div>
    """
)

w_transform = _make_text(
    description="Transform",
    placeholder="/path/to/transform.jsonl",
)

w_ref_path = _make_text(
    description="Large Img",
    placeholder="/path/to/large_reference.tif",
)

w_mov_path = _make_text(
    description="Crop Img",
    placeholder="/path/to/crop_moving.tif",
)

w_out_dir = _make_text(
    description="Out Dir",
    value=_default_output_dir(),
)

b_inspect = _make_button(
    description="Inspect & Validate",
    style="info",
    icon="search",
    width="99%",
    css_class="ns-action-button",
)

w_choice = widgets.Dropdown(
    options=[],
    description="Method",
    layout=widgets.Layout(width="98%", display="none"),
)

w_axes_ref = widgets.Combobox(
    options=[],
    description="Large Axes",
    layout=widgets.Layout(width="200px"),
)

w_pix_ref = widgets.FloatText(
    description="µm/px",
    layout=widgets.Layout(width="160px"),
)

w_axes_mov = widgets.Combobox(
    options=[],
    description="Crop Axes",
    layout=widgets.Layout(width="200px"),
)

w_pix_mov = widgets.FloatText(
    description="µm/px",
    layout=widgets.Layout(width="160px"),
)

w_margin = widgets.IntText(
    value=20,
    description="Pad (px)",
    layout=widgets.Layout(width="160px"),
)

w_confirm = widgets.Checkbox(
    value=False,
    description="I confirm metadata is correct",
    indent=False,
)

w_fullgrid_roi = widgets.Checkbox(
    value=True,
    description="Export",
    indent=False,
    layout=widgets.Layout(width="auto"),
)

w_fullgrid_full = widgets.Checkbox(
    value=False,
    description="Export",
    indent=False,
    layout=widgets.Layout(width="auto"),
)

w_cropgrid_roi = widgets.Checkbox(
    value=False,
    description="Export",
    indent=False,
    layout=widgets.Layout(width="auto"),
)

w_cropgrid_full = widgets.Checkbox(
    value=False,
    description="Export",
    indent=False,
    layout=widgets.Layout(width="auto"),
)

w_force_cropgrid = widgets.Checkbox(
    value=False,
    description="Force enable Crop Grid options",
)

w_force_cropgrid.layout.display = "none"

w_format = widgets.Dropdown(
    options=[
        ("TIFF (Standard)", "tiff"),
        ("OME-Zarr (High Performance)", "zarr"),
    ],
    value="tiff",
    description="Format",
    layout=widgets.Layout(width="300px"),
)

export_preview_html = widgets.HTML("")

status_html = widgets.HTML(
    "",
    layout=widgets.Layout(width="99%"),
)

w_log_toggle = widgets.ToggleButton(
    value=False,
    description="Show log",
    icon="terminal",
    tooltip="Show execution log",
    button_style="",
    layout=widgets.Layout(width="140px")
)

run_button = _make_button(
    description="Run Export",
    style="primary",
    width="99%",
    height="40px",
    css_class="ns-primary-button",
)

run_button.disabled = True

out = widgets.Output(
    layout=widgets.Layout(
        max_height="220px",
        overflow="auto",
        border="1px solid var(--ns-border)",
        padding="8px",
        display="none",
    )
)


# ============================================
# Widget styling
# ============================================

_desc_width(
    w_transform,
    w_ref_path,
    w_mov_path,
    w_out_dir,
    w="80px",
)

_desc_width(
    w_axes_ref,
    w_pix_ref,
    w_axes_mov,
    w_pix_mov,
    w_margin,
    w_choice,
    w_format,
    w="70px",
)


# ============================================
# UI builders
# ============================================

def _create_export_option(title, desc, svg, checkbox):
    """Create one export option card."""
    return widgets.HBox(
        [
            widgets.HTML(svg),
            widgets.VBox(
                [
                    widgets.HTML(
                        f"""
                        <div style='font-weight:700; font-size:12px; color:var(--ns-text);'>
                            {html.escape(title)}
                        </div>
                        """
                    ),
                    widgets.HTML(
                        f"""
                        <div style='font-size:11px; color:var(--ns-text-soft);
                                    margin-bottom:4px; line-height:1.2'>
                            {html.escape(desc)}
                        </div>
                        """
                    ),
                    checkbox,
                ]
            ),
        ],
        layout=widgets.Layout(
            border="1px solid var(--ns-border)",
            padding="8px",
            borderRadius="6px",
            width="99%",
            alignItems="center",
        ),
    )


def _build_inputs_card():
    return _make_card(
        [
            _make_section_header("1. Inputs"),
            w_transform,
            w_ref_path,
            w_mov_path,
            w_out_dir,
            widgets.Box(
                [b_inspect],
                layout=widgets.Layout(width="99%", margin="10px 0 0 0"),
            ),
            widgets.Box(
                [w_choice],
                layout=widgets.Layout(width="99%", margin="10px 0 0 0"),
            ),
        ],
        visible=True,
    )


def _build_review_card():
    content = widgets.VBox(
        [
            _make_section_header("2. Review Metadata"),
            widgets.HBox(
                [w_axes_ref, w_pix_ref],
                layout=widgets.Layout(margin="0 0 8px 0"),
            ),
            widgets.HBox(
                [w_axes_mov, w_pix_mov],
                layout=widgets.Layout(margin="0 0 12px 0"),
            ),
            widgets.HBox(
                [w_margin],
                layout=widgets.Layout(margin="0 0 12px 0"),
            ),
            w_confirm,
        ]
    )

    return _make_card([content], visible=False)


def _build_export_card():
    ui_full_roi = _create_export_option(
        "ROI on Reference Grid",
        "Overlays crop onto large image. Efficient.",
        _get_icon("ref", "roi"),
        w_fullgrid_roi,
    )

    ui_full_full = _create_export_option(
        "Full Canvas on Ref Grid",
        "Reconstructs entire space. Use Zarr for large images.",
        _get_icon("ref", "full"),
        w_fullgrid_full,
    )

    ui_crop_roi = _create_export_option(
        "ROI on Crop Grid",
        "Keeps original crop spacing.",
        _get_icon("crop", "roi"),
        w_cropgrid_roi,
    )

    ui_crop_full = _create_export_option(
        "Full Canvas on Crop Grid",
        "Resamples full image to crop grid.",
        _get_icon("crop", "full"),
        w_cropgrid_full,
    )

    STATE["ui_crop_roi"] = ui_crop_roi
    STATE["ui_crop_full"] = ui_crop_full

    return _make_card(
        [
            _make_section_header("3. Export Settings"),
            widgets.HBox([w_format]),
            _make_hr(),

            widgets.HTML("<div class='ns-subhead'>REFERENCE GRID</div>"),
            widgets.HBox(
                [ui_full_roi, ui_full_full],
                layout=widgets.Layout(gap="10px", margin="0 0 12px 0"),
            ),

            widgets.HTML("<div class='ns-subhead'>CROP GRID</div>"),
            widgets.HBox(
                [ui_crop_roi, ui_crop_full],
                layout=widgets.Layout(gap="10px", margin="0 0 12px 0"),
            ),

            w_force_cropgrid,
            _make_hr(),

            widgets.HTML(
                "<div style='font-size:12px; font-weight:700; color:var(--ns-label);'>"
                "Files to write:"
                "</div>"
            ),
            export_preview_html,
        ],
        visible=False,
    )


inputs_card = _build_inputs_card()
review_card = _build_review_card()
export_card = _build_export_card()


# ============================================
# UI logic
# ============================================

def _reset_ui(_=None):
    """Reset downstream UI after input paths change."""
    STATE.update(
        {
            "inspected": False,
            "candidates": [],
            "choice_to_record": {},
            "selected": None,
            "ref_header": None,
            "crop_header": None,
            "pix_same": None,
        }
    )

    review_card.layout.display = "none"
    export_card.layout.display = "none"

    w_choice.options = []
    w_choice.layout.display = "none"

    run_button.disabled = True

    status_html.value = ""
    export_preview_html.value = ""

    with out:
        clear_output()


def _update_export_matrix_state():
    """Enable/disable crop-grid exports depending on pixel-size compatibility."""
    if not STATE["inspected"]:
        return

    same_pixels = bool(STATE.get("pix_same"))
    crop_grid_allowed = (not same_pixels) or bool(w_force_cropgrid.value)

    ui_crop_roi = STATE.get("ui_crop_roi")
    ui_crop_full = STATE.get("ui_crop_full")

    opacity = "1.0" if crop_grid_allowed else "0.45"

    if ui_crop_roi is not None:
        ui_crop_roi.layout.opacity = opacity

    if ui_crop_full is not None:
        ui_crop_full.layout.opacity = opacity

    w_force_cropgrid.layout.display = "flex" if same_pixels else "none"

    w_cropgrid_roi.disabled = not crop_grid_allowed
    w_cropgrid_full.disabled = not crop_grid_allowed

    if not crop_grid_allowed:
        w_cropgrid_roi.value = False
        w_cropgrid_full.value = False


def _update_preview():
    """Update the list of files that will be exported."""
    ext = ".zarr" if w_format.value == "zarr" else ".tif"

    files = []

    if w_fullgrid_roi.value:
        files.append(f"__fullgrid__roiXY{ext}")

    if w_fullgrid_full.value:
        files.append(f"__fullgrid__fullXY{ext}")

    if w_cropgrid_roi.value:
        files.append(f"__cropgrid__roiXY{ext}")

    if w_cropgrid_full.value:
        files.append(f"__cropgrid__fullXY{ext}")

    if not files:
        export_preview_html.value = (
            "<div style='color:var(--ns-err-text); font-size:12px;'>"
            "No output selected."
            "</div>"
        )
        return

    export_preview_html.value = "".join(
        [
            (
                "<div style='font-family:monospace; font-size:11px; color:var(--ns-label);'>"
                f"• {html.escape(name)}"
                "</div>"
            )
            for name in files
        ]
    )


def _validation_reasons():
    reasons = []

    if not STATE["inspected"]:
        reasons.append("Inspection required.")

    if not w_confirm.value:
        reasons.append("Confirm metadata in the Review Metadata card.")

    has_export = any(
        [
            w_fullgrid_roi.value,
            w_fullgrid_full.value,
            w_cropgrid_roi.value,
            w_cropgrid_full.value,
        ]
    )

    if not has_export:
        reasons.append("Select at least one export output.")

    return reasons


def _validate(_=None):
    """Enable Run Export only when the workflow is ready."""
    reasons = _validation_reasons()
    ok = len(reasons) == 0

    run_button.disabled = not ok

    if STATE["inspected"] and not ok:
        status_html.value = _status_warn(reasons[0], title="Almost ready")

    return ok


def _set_selected_from_choice(_=None):
    """Update selected transform record when the user changes the dropdown."""
    label = w_choice.value

    if label in STATE["choice_to_record"]:
        STATE["selected"] = STATE["choice_to_record"][label]


def _populate_transform_choices(best, shortlist):
    """Populate dropdown with transform candidates."""
    STATE["candidates"] = shortlist
    STATE["choice_to_record"] = {}

    if len(shortlist) <= 1:
        w_choice.options = []
        w_choice.layout.display = "none"
        STATE["selected"] = best
        return

    labels = []

    for record in shortlist:
        label = _format_option(record)
        labels.append(label)
        STATE["choice_to_record"][label] = record

    best_label = _format_option(best)

    w_choice.options = labels
    w_choice.value = best_label if best_label in labels else labels[0]
    w_choice.layout.display = "flex"

    _set_selected_from_choice()


# ============================================
# Button callbacks
# ============================================

def _on_inspect_click(_):
    status_html.value = _status_info("Inspecting inputs and transform records.", title="Inspecting")

    with out:
        clear_output()

    missing = _preflight_missing()

    if missing:
        status_html.value = _status_err(missing, title="Missing helper functions")
        return

    try:
        transform_path = w_transform.value.strip()
        ref_path = w_ref_path.value.strip()
        mov_path = w_mov_path.value.strip()

        recs = load_transforms_any(transform_path)
        ref_header = inspect_image_header(ref_path)
        crop_header = inspect_image_header(mov_path)

        target_full_pix = ref_header.get("pixel_size_um") or (
            float(w_pix_ref.value) if w_pix_ref.value > 0 else None
        )

        target_crop_pix = crop_header.get("pixel_size_um") or (
            float(w_pix_mov.value) if w_pix_mov.value > 0 else None
        )

        best, shortlist = pick_best_transform(
            recs,
            target_full_pix,
            target_crop_pix,
            pix_rtol=0.02,
        )

        STATE.update(
            {
                "selected": best,
                "ref_header": ref_header,
                "crop_header": crop_header,
                "inspected": True,
            }
        )

        if best.get("pixel_size_full_um"):
            w_pix_ref.value = float(best["pixel_size_full_um"])
        elif ref_header.get("pixel_size_um"):
            w_pix_ref.value = float(ref_header["pixel_size_um"])

        if best.get("pixel_size_crop_um"):
            w_pix_mov.value = float(best["pixel_size_crop_um"])
        elif crop_header.get("pixel_size_um"):
            w_pix_mov.value = float(crop_header["pixel_size_um"])

        _set_axes_widget_from_header(w_axes_ref, ref_header)
        _set_axes_widget_from_header(w_axes_mov, crop_header)

        _populate_transform_choices(best, shortlist)

        STATE["pix_same"] = _pix_sizes_same(w_pix_ref.value, w_pix_mov.value)

        review_card.layout.display = "flex"
        export_card.layout.display = "flex"

        _update_export_matrix_state()
        _update_preview()

        status_html.value = _status_ok(
            ["Inspection successful. Review the metadata below before exporting."],
            title="Inspection complete",
        )

        _validate()

    except Exception as exc:
        _reset_ui()
        status_html.value = _status_err([str(exc)], title="Inspection failed")

        with out:
            traceback.print_exc()


def _on_run_click(_):
    if not _validate():
        return

    run_button.disabled = True
    status_html.value = _status_info("Running export.", title="Exporting")

    with out:
        clear_output()

        try:
            print("--- Starting Export ---")

            report = run_export_with_record(
                rec=STATE["selected"],
                transform_path=w_transform.value.strip(),
                ref_path=w_ref_path.value.strip(),
                mov_path=w_mov_path.value.strip(),
                out_dir=w_out_dir.value.strip(),
                axes_ref=str(w_axes_ref.value).strip(),
                axes_mov=str(w_axes_mov.value).strip(),
                pixel_size_ref_um=float(w_pix_ref.value),
                pixel_size_mov_um=float(w_pix_mov.value),
                margin_px=int(w_margin.value),
                order_intensity=1,
                pixel_size_equal_rtol=PIX_EQUAL_RTOL,
                export_fullgrid_fullXY=bool(w_fullgrid_full.value),
                export_fullgrid_roiXY=bool(w_fullgrid_roi.value),
                export_cropgrid_fullXY=bool(w_cropgrid_full.value),
                export_cropgrid_roiXY=bool(w_cropgrid_roi.value),
                force_cropgrid=bool(w_force_cropgrid.value),
                format=w_format.value,
            )

            files = report.get("files", [])

            print(f"Success. Wrote {len(files)} files/stores.")

            for file_path in files:
                print(f" - {Path(file_path).name}")

            status_html.value = _status_ok(
                [f"Export complete. {len(files)} item(s) written."],
                title="Export complete",
            )

        except Exception as exc:
            status_html.value = _status_err([str(exc)], title="Export failed")
            traceback.print_exc()

        finally:
            _validate()



def _on_metadata_change(_=None):
    """Keep pixel-size dependent UI state in sync after manual metadata edits."""
    if STATE["inspected"]:
        STATE["pix_same"] = _pix_sizes_same(w_pix_ref.value, w_pix_mov.value)

    _update_export_matrix_state()
    _update_preview()
    _validate()


# ============================================
# Observers
# ============================================

for widget in [w_transform, w_ref_path, w_mov_path]:
    widget.observe(_reset_ui, names="value")

def _on_log_toggle(change):
    show_log = bool(change["new"])

    _set_visible(out, show_log, display_mode="block")

    if show_log:
        w_log_toggle.description = "Hide log"
        w_log_toggle.icon = "eye-slash"
        w_log_toggle.tooltip = "Hide execution log"
    else:
        w_log_toggle.description = "Show log"
        w_log_toggle.icon = "terminal"
        w_log_toggle.tooltip = "Show execution log"


w_log_toggle.observe(_on_log_toggle, names="value")

for widget in [w_confirm, w_axes_ref, w_pix_ref, w_axes_mov, w_pix_mov, w_margin]:
    widget.observe(_on_metadata_change, names="value")

for widget in [
    w_fullgrid_roi,
    w_fullgrid_full,
    w_cropgrid_roi,
    w_cropgrid_full,
    w_force_cropgrid,
    w_format,
]:
    widget.observe(
        lambda _: (
            _update_export_matrix_state(),
            _update_preview(),
            _validate(),
        ),
        names="value",
    )

b_inspect.on_click(_on_inspect_click)
run_button.on_click(_on_run_click)


# ============================================
# Final layout
# ============================================

inner3 = widgets.VBox(
    [
        title,
        inputs_card,
        review_card,
        export_card,
        run_button,
        widgets.Box(
            [status_html],
            layout=widgets.Layout(width="99%", margin="10px 0"),
        ),
        widgets.HBox(
            [w_log_toggle],
            layout=widgets.Layout(
                width="99%",
                margin="16px 0 0 0",
                justify_content="center",
            )
        ),
        out,
    ]
)

inner3.add_class("ns-root")

ui = widgets.VBox(
    [
        style_html,
        inner3,
    ],
    layout=widgets.Layout(width="90%", margin="0 auto"),
)

display(ui)
_validate();